In [1]:
import torch
import torchvision

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

PyTorch: 2.14.0+cpu
Torchvision: 0.29.0+cpu
Device: cpu


In [2]:
import os

BASE_DIR = r"C:\Users\vu241\OneDrive\Desktop\Diabetic-Retinopathy-Screening\Disease Dataset\B. Disease Grading"

print("Dataset exists:", os.path.exists(BASE_DIR))

Dataset exists: True


In [3]:
TRAIN_DIR = os.path.join(
    BASE_DIR,
    "1. Original Images",
    "a. Training Set"
)

print("Training folder exists:", os.path.exists(TRAIN_DIR))
print("Training folder:", TRAIN_DIR)

Training folder exists: True
Training folder: C:\Users\vu241\OneDrive\Desktop\Diabetic-Retinopathy-Screening\Disease Dataset\B. Disease Grading\1. Original Images\a. Training Set


In [4]:
files = os.listdir(TRAIN_DIR)

print("Total files:", len(files))
print("\nFirst 10 files:")

for file in files[:10]:
    print(file)

Total files: 413

First 10 files:
IDRiD_001.jpg
IDRiD_002.jpg
IDRiD_003.jpg
IDRiD_004.jpg
IDRiD_005.jpg
IDRiD_006.jpg
IDRiD_007.jpg
IDRiD_008.jpg
IDRiD_009.jpg
IDRiD_010.jpg


In [5]:
GROUNDTRUTH_DIR = os.path.join(
    BASE_DIR,
    "2. Groundtruths"
)

print("Groundtruth folder exists:", os.path.exists(GROUNDTRUTH_DIR))

print("\nFiles inside Groundtruths:")
for file in os.listdir(GROUNDTRUTH_DIR):
    print(file)

Groundtruth folder exists: True

Files inside Groundtruths:
a. IDRiD_Disease Grading_Training Labels.csv
b. IDRiD_Disease Grading_Testing Labels.csv


In [6]:
import pandas as pd

In [7]:
# Remove extra spaces from column names
df.columns = df.columns.str.strip()

print(df.columns.tolist())

NameError: name 'df' is not defined

In [ ]:
# Keep only the columns we need
df = df[[
    "Image name",
    "Retinopathy grade",
    "Risk of macular edema"
]]

print("Cleaned dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 10 rows:")
display(df.head(10))

In [ ]:
import glob

# Get all image files
image_files = glob.glob(os.path.join(TRAIN_DIR, "*.jpg"))

# Create mapping: IDRiD_001 -> full image path
image_map = {}

for img in image_files:
    name = os.path.splitext(os.path.basename(img))[0]
    image_map[name] = img

# Add image path to dataframe
df["image_path"] = df["Image name"].map(image_map)

print("Total images found:", len(image_map))
print("Missing image paths:", df["image_path"].isna().sum())

display(df.head())

In [ ]:
# Count images in each Retinopathy Grade
class_counts = df["Retinopathy grade"].value_counts().sort_index()

print("Retinopathy Grade Distribution:")
print(class_counts)

In [ ]:
class_percentage = (
    df["Retinopathy grade"]
    .value_counts(normalize=True)
    .sort_index()
    * 100
)

print("\nClass Percentage:")
print(class_percentage.round(2))

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

In [ ]:
# Display one sample fundus image

sample = df.iloc[0]

img = Image.open(sample["image_path"])

plt.figure(figsize=(8, 8))
plt.imshow(img)
plt.axis("off")

plt.title(
    f"{sample['Image name']} | "
    f"DR Grade: {sample['Retinopathy grade']}"
)

plt.show()

print("Image size:", img.size)
print("Image mode:", img.mode)

In [ ]:
# Display one sample image from each DR grade

grades = [0, 1, 2, 3, 4]

plt.figure(figsize=(15, 10))

for i, grade in enumerate(grades):
    sample = df[df["Retinopathy grade"] == grade].iloc[0]
    
    img = Image.open(sample["image_path"])
    
    plt.subplot(2, 3, i + 1)
    plt.imshow(img)
    plt.axis("off")
    plt.title(
        f"Grade {grade}\n{sample['Image name']}"
    )

plt.tight_layout()
plt.show()

In [ ]:
from torchvision import transforms

# Basic preprocessing
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

sample = df.iloc[0]

img = Image.open(sample["image_path"]).convert("RGB")

processed_img = transform(img)

print("Original image size:", img.size)
print("Processed tensor shape:", processed_img.shape)
print("Tensor data type:", processed_img.dtype)

In [ ]:
from sklearn.model_selection import train_test_split

# Features and labels
X = df["image_path"]
y = df["Retinopathy grade"]

# Stratified 80/20 split
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training images:", len(X_train))
print("Validation images:", len(X_val))

print("\nTraining class distribution:")
print(y_train.value_counts().sort_index())

print("\nValidation class distribution:")
print(y_val.value_counts().sort_index())

In [ ]:
# Create training dataframe
train_df = pd.DataFrame({
    "image_path": X_train.values,
    "label": y_train.values
})

# Create validation dataframe
val_df = pd.DataFrame({
    "image_path": X_val.values,
    "label": y_val.values
})

# Reset indexes
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("Training DataFrame:")
display(train_df.head())

print("\nValidation DataFrame:")
display(val_df.head())

print("\nShapes:")
print("Train:", train_df.shape)
print("Validation:", val_df.shape)

In [ ]:
import torch
from torch.utils.data import Dataset


class RetinaDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):

        # Get image path and label
        image_path = self.dataframe.loc[index, "image_path"]
        label = self.dataframe.loc[index, "label"]

        # Open image
        image = Image.open(image_path).convert("RGB")

        # Apply transform
        if self.transform:
            image = self.transform(image)

        # Convert label to tensor
        label = torch.tensor(label, dtype=torch.long)

        return image, label

In [ ]:
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

test_dataset = RetinaDataset(
    train_df,
    transform=test_transform
)

print("Dataset length:", len(test_dataset))

In [ ]:
from torchvision import transforms

# Training transformations
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Validation transformations
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("Training transform:")
print(train_transform)

print("\nValidation transform:")
print(val_transform)

In [ ]:
# Create training dataset
train_dataset = RetinaDataset(
    train_df,
    transform=train_transform
)

# Create validation dataset
val_dataset = RetinaDataset(
    val_df,
    transform=val_transform
)

print("Training dataset size:", len(train_dataset))
print("Validation dataset size:", len(val_dataset))

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))

In [ ]:
images, labels = next(iter(train_loader))

print("Batch image shape:", images.shape)
print("Batch labels shape:", labels.shape)
print("Labels:", labels)

In [ ]:
# EfficientNet-B0 Setup

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

# Load pretrained EfficientNet-B0
weights = EfficientNet_B0_Weights.DEFAULT
model = efficientnet_b0(weights=weights)

# Replace the final classifier
num_features = model.classifier[1].in_features

model.classifier[1] = nn.Linear(
    num_features,
    5
)

# Use CPU for now
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

print("Device:", device)
print("Model: EfficientNet-B0")
print("Number of classes:", 5)
print("Classifier:", model.classifier)

In [ ]:
import numpy as np
import torch

# Count samples for each class in training data
class_counts = (
    train_df["label"]
    .value_counts()
    .sort_index()
)

print("Training class counts:")
print(class_counts)

# Calculate balanced class weights
num_classes = 5
total_samples = len(train_df)

class_weights = total_samples / (
    num_classes * class_counts.values
)

print("\nClass weights:")
for grade, weight in enumerate(class_weights):
    print(f"Grade {grade}: {weight:.4f}")

# Convert to PyTorch tensor
class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32
).to(device)

print("\nPyTorch class weights:")
print(class_weights)

In [ ]:
import torch.optim as optim

# Weighted Cross Entropy Loss
criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

# Adam optimizer
optimizer = optim.Adam(
    model.parameters(),
    lr=0.0001
)

print("Loss function:")
print(criterion)

print("\nOptimizer:")
print(optimizer)

In [ ]:
# Get one batch
images, labels = next(iter(train_loader))

# Move batch to device
images = images.to(device)
labels = labels.to(device)

# Forward pass
outputs = model(images)

# Calculate loss
loss = criterion(outputs, labels)

print("Input shape:", images.shape)
print("Output shape:", outputs.shape)
print("Labels shape:", labels.shape)
print("Loss:", loss.item())

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:

        images = images.to(device)
        labels = labels.to(device)

        # Clear old gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)

        # Calculate loss
        loss = criterion(outputs, labels)

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        # Statistics
        running_loss += loss.item() * images.size(0)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_accuracy = correct / total

    return epoch_loss, epoch_accuracy

In [ ]:
train_loss, train_accuracy = train_one_epoch(
    model,
    train_loader,
    criterion,
    optimizer,
    device
)

print(f"Training Loss: {train_loss:.4f}")
print(f"Training Accuracy: {train_accuracy * 100:.2f}%")

In [ ]:
def validate(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            # Forward pass
            outputs = model(images)

            # Calculate loss
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            # Predictions
            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_accuracy = correct / total

    return epoch_loss, epoch_accuracy

In [ ]:
val_loss, val_accuracy = validate(
    model,
    val_loader,
    criterion,
    device
)

print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy * 100:.2f}%")

In [ ]:
import copy

NUM_EPOCHS = 5

best_val_loss = float("inf")
best_model_state = copy.deepcopy(model.state_dict())

history = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": []
}

for epoch in range(NUM_EPOCHS):

    print(f"\nEpoch {epoch + 1}/{NUM_EPOCHS}")
    print("-" * 40)

    # Training
    train_loss, train_accuracy = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    # Validation
    val_loss, val_accuracy = validate(
        model,
        val_loader,
        criterion,
        device
    )

    # Save history
    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_accuracy)
    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_accuracy)

    print(f"Train Loss:      {train_loss:.4f}")
    print(f"Train Accuracy:  {train_accuracy * 100:.2f}%")
    print(f"Val Loss:        {val_loss:.4f}")
    print(f"Val Accuracy:    {val_accuracy * 100:.2f}%")

    # Save best model in memory
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = copy.deepcopy(model.state_dict())

        print("✅ Best model updated!")

# Restore best model
model.load_state_dict(best_model_state)

print("\nTraining complete!")
print(f"Best Validation Loss: {best_val_loss:.4f}")

In [ ]:
import os
import torch

# Create models directory
os.makedirs("models", exist_ok=True)

# Save best model
MODEL_PATH = "models/retina_xai_efficientnet_b0.pth"

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "num_classes": 5,
        "class_names": {
            0: "No DR",
            1: "Mild",
            2: "Moderate",
            3: "Severe",
            4: "Proliferative"
        },
        "best_val_loss": best_val_loss
    },
    MODEL_PATH
)

print("Model saved successfully!")
print("Path:", MODEL_PATH)
print("File exists:", os.path.exists(MODEL_PATH))

In [ ]:
# Load the saved checkpoint

checkpoint = torch.load(
    MODEL_PATH,
    map_location=device
)

# Load model weights
model.load_state_dict(
    checkpoint["model_state_dict"]
)

model = model.to(device)

print("Model reloaded successfully!")
print("Number of classes:", checkpoint["num_classes"])
print("Best validation loss:", checkpoint["best_val_loss"])
print("Model file exists:", os.path.exists(MODEL_PATH))

In [ ]:
# Testing image directory

TEST_DIR = os.path.join(
    BASE_DIR,
    "1. Original Images",
    "b. Testing Set"
)

print("Testing directory exists:", os.path.exists(TEST_DIR))

test_image_files = glob.glob(
    os.path.join(TEST_DIR, "*.jpg")
)

print("Total testing images:", len(test_image_files))

print("\nFirst 10 testing images:")
print([
    os.path.basename(x)
    for x in test_image_files[:10]
])

In [ ]:
import os

print("Files inside Groundtruths folder:\n")

for file in os.listdir(GROUNDTRUTH_DIR):
    print(repr(file))

In [ ]:
TEST_LABEL_FILE = os.path.join(
    GROUNDTRUTH_DIR,
    "b. IDRiD_Disease Grading_Testing Labels.csv"
)

test_df = pd.read_csv(TEST_LABEL_FILE)

# Clean column names
test_df.columns = test_df.columns.str.strip()

print("Testing dataset shape:", test_df.shape)

print("\nColumns:")
print(test_df.columns.tolist())

print("\nFirst 10 rows:")
display(test_df.head(10))

In [ ]:
# Create mapping for testing images

test_image_files = glob.glob(
    os.path.join(TEST_DIR, "*.jpg")
)

test_image_map = {}

for img in test_image_files:
    name = os.path.splitext(
        os.path.basename(img)
    )[0]
    
    test_image_map[name] = img

# Add image paths to testing dataframe
test_df["image_path"] = test_df["Image name"].map(test_image_map)

print("Testing images found:", len(test_image_map))
print("Testing labels:", len(test_df))
print("Missing image paths:", test_df["image_path"].isna().sum())

display(test_df.head())

In [ ]:
# Create testing dataset

test_dataset = RetinaDataset(
    test_df,
    transform=val_transform
)

print("Testing dataset size:", len(test_dataset))

In [ ]:
test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0
)

print("Testing batches:", len(test_loader))

In [ ]:
# Create the label column for the testing dataset

test_df["label"] = test_df["Retinopathy grade"]

print("Testing columns:")
print(test_df.columns.tolist())

display(test_df.head())

In [ ]:
test_dataset = RetinaDataset(
    test_df,
    transform=val_transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0
)

print("Testing dataset size:", len(test_dataset))
print("Testing batches:", len(test_loader))

In [ ]:
model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = model(images)

        # Predicted class
        _, predictions = torch.max(outputs, 1)

        # Store predictions and actual labels
        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_labels.extend(
            labels.cpu().numpy()
        )

print("Total predictions:", len(all_predictions))
print("Total actual labels:", len(all_labels))

print("\nFirst 20 predictions:")
print(all_predictions[:20])

print("\nFirst 20 actual labels:")
print(all_labels[:20])

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report
)

# Test accuracy
test_accuracy = accuracy_score(
    all_labels,
    all_predictions
)

print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

# Classification report
class_names = [
    "No DR",
    "Mild",
    "Moderate",
    "Severe",
    "Proliferative"
]

print("\nClassification Report:\n")

print(
    classification_report(
        all_labels,
        all_predictions,
        labels=[0, 1, 2, 3, 4],
        target_names=class_names,
        zero_division=0
    )
)

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

cm = confusion_matrix(
    all_labels,
    all_predictions,
    labels=[0, 1, 2, 3, 4]
)

class_names = [
    "No DR",
    "Mild",
    "Moderate",
    "Severe",
    "Proliferative"
]

plt.figure(figsize=(8, 6))

plt.imshow(cm)

plt.title("Confusion Matrix - DR Classification")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")

plt.xticks(
    range(5),
    class_names,
    rotation=45
)

plt.yticks(
    range(5),
    class_names
)

# Display numbers inside cells
for i in range(5):
    for j in range(5):
        plt.text(
            j,
            i,
            cm[i, j],
            ha="center",
            va="center"
        )

plt.colorbar()
plt.tight_layout()
plt.show()

In [ ]:
from torch.utils.data import WeightedRandomSampler
import torch.nn as nn
import torch.optim as optim

In [ ]:
# Count samples of each class
class_counts = train_df["label"].value_counts().sort_index()

print("Training class counts:")
print(class_counts)

# Weight for each class
class_sample_weights = 1.0 / class_counts.values

print("\nClass sampling weights:")
for i, weight in enumerate(class_sample_weights):
    print(f"Class {i}: {weight:.4f}")

In [ ]:
# Weight for every individual training image
sample_weights = train_df["label"].map(
    lambda label: class_sample_weights[label]
).values

sample_weights = torch.tensor(
    sample_weights,
    dtype=torch.double
)

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

print("Weighted sampler created successfully!")

In [ ]:
BATCH_SIZE = 16

train_loader_balanced = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("Balanced training batches:", len(train_loader_balanced))
print("Validation batches:", len(val_loader))

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

# Load pretrained EfficientNet-B0
weights = EfficientNet_B0_Weights.DEFAULT

improved_model = efficientnet_b0(weights=weights)

# Replace final classifier
num_features = improved_model.classifier[1].in_features

improved_model.classifier[1] = nn.Linear(
    num_features,
    5
)

# Device
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

improved_model = improved_model.to(device)

print("Improved model created successfully!")
print("Device:", device)
print("Output classes:", 5)

In [ ]:
criterion_improved = nn.CrossEntropyLoss()

optimizer_improved = optim.Adam(
    improved_model.parameters(),
    lr=0.0001
)

print("Loss and optimizer configured!")

In [ ]:
NUM_EPOCHS = 10

best_val_loss_improved = float("inf")
best_model_state_improved = None

history_improved = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": []
}

for epoch in range(NUM_EPOCHS):

    print(f"\nEpoch {epoch + 1}/{NUM_EPOCHS}")
    print("-" * 40)

    # Training
    train_loss, train_accuracy = train_one_epoch(
        improved_model,
        train_loader_balanced,
        criterion_improved,
        optimizer_improved,
        device
    )

    # Validation
    val_loss, val_accuracy = validate(
        improved_model,
        val_loader,
        criterion_improved,
        device
    )

    # Store history
    history_improved["train_loss"].append(train_loss)
    history_improved["train_accuracy"].append(train_accuracy)

    history_improved["val_loss"].append(val_loss)
    history_improved["val_accuracy"].append(val_accuracy)

    # Print results
    print(f"Train Loss:      {train_loss:.4f}")
    print(f"Train Accuracy:  {train_accuracy * 100:.2f}%")
    print(f"Val Loss:        {val_loss:.4f}")
    print(f"Val Accuracy:    {val_accuracy * 100:.2f}%")

    # Save best model
    if val_loss < best_val_loss_improved:

        best_val_loss_improved = val_loss

        best_model_state_improved = {
            key: value.cpu().clone()
            for key, value in improved_model.state_dict().items()
        }

        print("✅ Best improved model updated!")

# Load best model
improved_model.load_state_dict(best_model_state_improved)

improved_model = improved_model.to(device)

print("\n" + "=" * 50)
print("Improved training complete!")
print(f"Best Validation Loss: {best_val_loss_improved:.4f}")
print("=" * 50)

In [ ]:
import os
import torch

os.makedirs("models", exist_ok=True)

IMPROVED_MODEL_PATH = "models/retina_xai_efficientnet_b0_improved.pth"

torch.save({
    "model_state_dict": improved_model.state_dict(),
    "num_classes": 5,
    "class_names": {
        0: "No DR",
        1: "Mild",
        2: "Moderate",
        3: "Severe",
        4: "Proliferative"
    },
    "best_val_loss": best_val_loss_improved
}, IMPROVED_MODEL_PATH)

print("Improved model saved successfully!")
print("Path:", IMPROVED_MODEL_PATH)
print("File exists:", os.path.exists(IMPROVED_MODEL_PATH))

In [ ]:
improved_model.eval()

improved_predictions = []
test_labels = []

with torch.no_grad():
    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = improved_model(images)

        _, predictions = torch.max(outputs, 1)

        improved_predictions.extend(
            predictions.cpu().numpy()
        )

        test_labels.extend(
            labels.cpu().numpy()
        )

print("Total predictions:", len(improved_predictions))
print("Total actual labels:", len(test_labels))

print("\nFirst 20 improved predictions:")
print(improved_predictions[:20])

print("\nFirst 20 actual labels:")
print(test_labels[:20])

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

# Test accuracy
improved_test_accuracy = accuracy_score(
    test_labels,
    improved_predictions
)

print(f"Improved Test Accuracy: {improved_test_accuracy * 100:.2f}%")

# Classification report
class_names = [
    "No DR",
    "Mild",
    "Moderate",
    "Severe",
    "Proliferative"
]

print("\nImproved Classification Report:\n")

print(
    classification_report(
        test_labels,
        improved_predictions,
        labels=[0, 1, 2, 3, 4],
        target_names=class_names,
        zero_division=0
    )
)

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

cm_improved = confusion_matrix(
    test_labels,
    improved_predictions,
    labels=[0, 1, 2, 3, 4]
)

class_names = [
    "No DR",
    "Mild",
    "Moderate",
    "Severe",
    "Proliferative"
]

plt.figure(figsize=(8, 6))

plt.imshow(cm_improved)

plt.title("Improved Model - Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")

plt.xticks(
    range(5),
    class_names,
    rotation=45
)

plt.yticks(
    range(5),
    class_names
)

for i in range(5):
    for j in range(5):
        plt.text(
            j,
            i,
            cm_improved[i, j],
            ha="center",
            va="center"
        )

plt.colorbar()
plt.tight_layout()
plt.show()

In [ ]:
epochs = range(1, len(history_improved["train_loss"]) + 1)

# Loss curve
plt.figure(figsize=(8, 5))

plt.plot(
    epochs,
    history_improved["train_loss"],
    marker="o",
    label="Training Loss"
)

plt.plot(
    epochs,
    history_improved["val_loss"],
    marker="o",
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    epochs,
    [x * 100 for x in history_improved["train_accuracy"]],
    marker="o",
    label="Training Accuracy"
)

plt.plot(
    epochs,
    [x * 100 for x in history_improved["val_accuracy"]],
    marker="o",
    label="Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("Training vs Validation Accuracy")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
import torch
import torch.nn as nn
import torch.optim as optim

weights = EfficientNet_B0_Weights.DEFAULT

finetune_model = efficientnet_b0(weights=weights)

# Replace classifier
num_features = finetune_model.classifier[1].in_features

finetune_model.classifier[1] = nn.Linear(
    num_features,
    5
)

finetune_model = finetune_model.to(device)

print("Fine-tuning model created!")
print("Device:", device)

In [ ]:
# Freeze EfficientNet feature extractor
for param in finetune_model.features.parameters():
    param.requires_grad = False

# Keep classifier trainable
for param in finetune_model.classifier.parameters():
    param.requires_grad = True

trainable_params = sum(
    p.numel()
    for p in finetune_model.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel()
    for p in finetune_model.parameters()
)

print("Trainable parameters:", trainable_params)
print("Total parameters:", total_params)

In [ ]:
criterion_finetune = nn.CrossEntropyLoss()

optimizer_finetune = optim.Adam(
    filter(
        lambda p: p.requires_grad,
        finetune_model.parameters()
    ),
    lr=0.001
)

print("Classifier training configured!")

In [ ]:
for epoch in range(3):

    print(f"\nClassifier Epoch {epoch + 1}/3")
    print("-" * 40)

    train_loss, train_accuracy = train_one_epoch(
        finetune_model,
        train_loader_balanced,
        criterion_finetune,
        optimizer_finetune,
        device
    )

    val_loss, val_accuracy = validate(
        finetune_model,
        val_loader,
        criterion_finetune,
        device
    )

    print(f"Train Loss:     {train_loss:.4f}")
    print(f"Train Accuracy: {train_accuracy * 100:.2f}%")
    print(f"Val Loss:       {val_loss:.4f}")
    print(f"Val Accuracy:   {val_accuracy * 100:.2f}%")

In [ ]:
# First keep all feature layers frozen
for param in finetune_model.features.parameters():
    param.requires_grad = False

# Unfreeze the last 2 feature blocks
for param in finetune_model.features[-2:].parameters():
    param.requires_grad = True

# Classifier remains trainable
for param in finetune_model.classifier.parameters():
    param.requires_grad = True

trainable_params = sum(
    p.numel()
    for p in finetune_model.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel()
    for p in finetune_model.parameters()
)

print("Trainable parameters:", trainable_params)
print("Total parameters:", total_params)

In [ ]:
optimizer_finetune = optim.Adam(
    filter(
        lambda p: p.requires_grad,
        finetune_model.parameters()
    ),
    lr=0.00001
)

criterion_finetune = nn.CrossEntropyLoss()

print("Fine-tuning optimizer configured!")
print("Learning rate: 0.00001")

In [ ]:
NUM_FINE_TUNE_EPOCHS = 5

best_ft_val_loss = float("inf")
best_ft_model_state = None

history_finetune = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": []
}

for epoch in range(NUM_FINE_TUNE_EPOCHS):

    print(f"\nFine-Tuning Epoch {epoch + 1}/{NUM_FINE_TUNE_EPOCHS}")
    print("-" * 45)

    train_loss, train_accuracy = train_one_epoch(
        finetune_model,
        train_loader_balanced,
        criterion_finetune,
        optimizer_finetune,
        device
    )

    val_loss, val_accuracy = validate(
        finetune_model,
        val_loader,
        criterion_finetune,
        device
    )

    history_finetune["train_loss"].append(train_loss)
    history_finetune["train_accuracy"].append(train_accuracy)
    history_finetune["val_loss"].append(val_loss)
    history_finetune["val_accuracy"].append(val_accuracy)

    print(f"Train Loss:     {train_loss:.4f}")
    print(f"Train Accuracy: {train_accuracy * 100:.2f}%")
    print(f"Val Loss:       {val_loss:.4f}")
    print(f"Val Accuracy:   {val_accuracy * 100:.2f}%")

    if val_loss < best_ft_val_loss:

        best_ft_val_loss = val_loss

        best_ft_model_state = {
            key: value.cpu().clone()
            for key, value in finetune_model.state_dict().items()
        }

        print("✅ Best fine-tuned model updated!")

# Restore best model
finetune_model.load_state_dict(best_ft_model_state)
finetune_model = finetune_model.to(device)

print("\n" + "=" * 55)
print("Fine-tuning complete!")
print(f"Best Validation Loss: {best_ft_val_loss:.4f}")
print("=" * 55)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

def preprocess_fundus(image_path, size=224):

    # Read image
    image = cv2.imread(image_path)

    # OpenCV BGR → RGB
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Grayscale
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

    # Threshold to detect retinal region
    _, mask = cv2.threshold(
        gray,
        10,
        255,
        cv2.THRESH_BINARY
    )

    # Find largest contour
    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    if contours:

        largest_contour = max(
            contours,
            key=cv2.contourArea
        )

        x, y, w, h = cv2.boundingRect(
            largest_contour
        )

        # Crop retinal region
        image = image[y:y+h, x:x+w]

    # Resize
    image = cv2.resize(
        image,
        (size, size)
    )

    return image

In [ ]:
sample_path = train_df.iloc[0]["image_path"]

original = Image.open(sample_path).convert("RGB")
processed = preprocess_fundus(sample_path)

plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.imshow(original)
plt.title("Original Fundus")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(processed)
plt.title("Processed Fundus")
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
class PreprocessedRetinaDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):

        image_path = self.dataframe.loc[index, "image_path"]
        label = self.dataframe.loc[index, "label"]

        # Fundus preprocessing
        image = preprocess_fundus(image_path)

        # NumPy → PIL
        image = Image.fromarray(image)

        # Existing augmentation + normalization
        if self.transform:
            image = self.transform(image)

        label = torch.tensor(
            label,
            dtype=torch.long
        )

        return image, label

In [ ]:
train_dataset_preprocessed = PreprocessedRetinaDataset(
    train_df,
    transform=train_transform
)

val_dataset_preprocessed = PreprocessedRetinaDataset(
    val_df,
    transform=val_transform
)

print("Training samples:", len(train_dataset_preprocessed))
print("Validation samples:", len(val_dataset_preprocessed))

In [ ]:
train_loader_preprocessed = DataLoader(
    train_dataset_preprocessed,
    batch_size=16,
    sampler=sampler,
    num_workers=0
)

val_loader_preprocessed = DataLoader(
    val_dataset_preprocessed,
    batch_size=16,
    shuffle=False,
    num_workers=0
)

print("Preprocessed training batches:", len(train_loader_preprocessed))
print("Preprocessed validation batches:", len(val_loader_preprocessed))

In [ ]:
test_dataset_preprocessed = PreprocessedRetinaDataset(
    test_df,
    transform=val_transform
)

test_loader_preprocessed = DataLoader(
    test_dataset_preprocessed,
    batch_size=16,
    shuffle=False,
    num_workers=0
)

print("Preprocessed test samples:", len(test_dataset_preprocessed))
print("Preprocessed test batches:", len(test_loader_preprocessed))

In [ ]:
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
import torch
import torch.nn as nn
import torch.optim as optim

weights = EfficientNet_B0_Weights.DEFAULT

preprocessed_model = efficientnet_b0(
    weights=weights
)

# Replace final classifier
num_features = preprocessed_model.classifier[1].in_features

preprocessed_model.classifier[1] = nn.Linear(
    num_features,
    5
)

preprocessed_model = preprocessed_model.to(device)

print("Preprocessed EfficientNet-B0 created!")
print("Device:", device)
print("Classes:", 5)

In [ ]:
criterion_preprocessed = nn.CrossEntropyLoss()

optimizer_preprocessed = optim.Adam(
    preprocessed_model.parameters(),
    lr=0.0001
)

print("Optimizer configured!")
print("Learning rate:", 0.0001)

In [ ]:
NUM_EPOCHS = 10

best_preprocessed_val_loss = float("inf")
best_preprocessed_state = None

history_preprocessed = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": []
}

for epoch in range(NUM_EPOCHS):

    print(f"\nEpoch {epoch + 1}/{NUM_EPOCHS}")
    print("-" * 40)

    train_loss, train_accuracy = train_one_epoch(
        preprocessed_model,
        train_loader_preprocessed,
        criterion_preprocessed,
        optimizer_preprocessed,
        device
    )

    val_loss, val_accuracy = validate(
        preprocessed_model,
        val_loader_preprocessed,
        criterion_preprocessed,
        device
    )

    history_preprocessed["train_loss"].append(train_loss)
    history_preprocessed["train_accuracy"].append(train_accuracy)

    history_preprocessed["val_loss"].append(val_loss)
    history_preprocessed["val_accuracy"].append(val_accuracy)

    print(f"Train Loss:      {train_loss:.4f}")
    print(f"Train Accuracy:  {train_accuracy * 100:.2f}%")
    print(f"Val Loss:        {val_loss:.4f}")
    print(f"Val Accuracy:    {val_accuracy * 100:.2f}%")

    if val_loss < best_preprocessed_val_loss:

        best_preprocessed_val_loss = val_loss

        best_preprocessed_state = {
            key: value.cpu().clone()
            for key, value in preprocessed_model.state_dict().items()
        }

        print("✅ Best preprocessed model updated!")

# Restore best model
preprocessed_model.load_state_dict(
    best_preprocessed_state
)

preprocessed_model = preprocessed_model.to(device)

print("\n" + "=" * 50)
print("Preprocessed training complete!")
print(
    f"Best Validation Loss: "
    f"{best_preprocessed_val_loss:.4f}"
)
print("=" * 50)

In [ ]:
preprocessed_model.eval()

preprocessed_predictions = []
preprocessed_labels = []

with torch.no_grad():

    for images, labels in test_loader_preprocessed:

        images = images.to(device)
        labels = labels.to(device)

        outputs = preprocessed_model(images)

        _, predictions = torch.max(outputs, 1)

        preprocessed_predictions.extend(
            predictions.cpu().numpy()
        )

        preprocessed_labels.extend(
            labels.cpu().numpy()
        )

print("Total predictions:", len(preprocessed_predictions))
print("Total actual labels:", len(preprocessed_labels))

In [ ]:
from sklearn.metrics import accuracy_score

preprocessed_test_accuracy = accuracy_score(
    preprocessed_labels,
    preprocessed_predictions
)

print(
    f"Preprocessed Test Accuracy: "
    f"{preprocessed_test_accuracy * 100:.2f}%"
)

In [ ]:
from sklearn.metrics import classification_report

class_names = [
    "No DR",
    "Mild",
    "Moderate",
    "Severe",
    "Proliferative"
]

print("Preprocessed Classification Report:\n")

print(
    classification_report(
        preprocessed_labels,
        preprocessed_predictions,
        labels=[0, 1, 2, 3, 4],
        target_names=class_names,
        zero_division=0
    )
)

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

cm_preprocessed = confusion_matrix(
    preprocessed_labels,
    preprocessed_predictions,
    labels=[0, 1, 2, 3, 4]
)

class_names = [
    "No DR",
    "Mild",
    "Moderate",
    "Severe",
    "Proliferative"
]

plt.figure(figsize=(8, 6))

plt.imshow(cm_preprocessed)

plt.title("Preprocessed Model - Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")

plt.xticks(
    range(5),
    class_names,
    rotation=45
)

plt.yticks(
    range(5),
    class_names
)

for i in range(5):
    for j in range(5):
        plt.text(
            j,
            i,
            cm_preprocessed[i, j],
            ha="center",
            va="center"
        )

plt.colorbar()
plt.tight_layout()
plt.show()

In [ ]:
import os
import torch

os.makedirs("models", exist_ok=True)

FINAL_MODEL_PATH = "models/retina_xai_final_efficientnet_b0.pth"

torch.save({
    "model_state_dict": preprocessed_model.state_dict(),
    "num_classes": 5,
    "class_names": {
        0: "No DR",
        1: "Mild",
        2: "Moderate",
        3: "Severe",
        4: "Proliferative"
    },
    "best_val_loss": best_preprocessed_val_loss,
    "test_accuracy": preprocessed_test_accuracy
}, FINAL_MODEL_PATH)

print("Final model saved successfully!")
print("Path:", FINAL_MODEL_PATH)
print("File exists:", os.path.exists(FINAL_MODEL_PATH))
print(
    f"Test Accuracy: "
    f"{preprocessed_test_accuracy * 100:.2f}%"
)

In [ ]:
train_transform_300 = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.15
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform_300 = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("300x300 transforms created!")

In [ ]:
train_dataset_300 = PreprocessedRetinaDataset(
    train_df,
    transform=train_transform_300
)

val_dataset_300 = PreprocessedRetinaDataset(
    val_df,
    transform=val_transform_300
)

test_dataset_300 = PreprocessedRetinaDataset(
    test_df,
    transform=val_transform_300
)

print("Train:", len(train_dataset_300))
print("Validation:", len(val_dataset_300))
print("Test:", len(test_dataset_300))

In [ ]:
train_loader_300 = DataLoader(
    train_dataset_300,
    batch_size=16,
    sampler=sampler,
    num_workers=0
)

val_loader_300 = DataLoader(
    val_dataset_300,
    batch_size=16,
    shuffle=False,
    num_workers=0
)

test_loader_300 = DataLoader(
    test_dataset_300,
    batch_size=16,
    shuffle=False,
    num_workers=0
)

print("Training batches:", len(train_loader_300))
print("Validation batches:", len(val_loader_300))
print("Test batches:", len(test_loader_300))

In [ ]:
images, labels = next(iter(train_loader_300))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Sample labels:", labels[:10])

In [ ]:
weights = EfficientNet_B0_Weights.DEFAULT

model_300 = efficientnet_b0(
    weights=weights
)

num_features = model_300.classifier[1].in_features

model_300.classifier[1] = nn.Linear(
    num_features,
    5
)

model_300 = model_300.to(device)

print("300x300 EfficientNet-B0 created!")
print("Device:", device)
print("Output classes:", 5)

In [ ]:
criterion_300 = nn.CrossEntropyLoss()

optimizer_300 = optim.Adam(
    model_300.parameters(),
    lr=0.0001
)

print("Optimizer configured!")

In [ ]:
NUM_EPOCHS_300 = 10

best_val_loss_300 = float("inf")
best_model_state_300 = None

history_300 = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": []
}

for epoch in range(NUM_EPOCHS_300):

    print(f"\nEpoch {epoch + 1}/{NUM_EPOCHS_300}")
    print("-" * 40)

    train_loss, train_accuracy = train_one_epoch(
        model_300,
        train_loader_300,
        criterion_300,
        optimizer_300,
        device
    )

    val_loss, val_accuracy = validate(
        model_300,
        val_loader_300,
        criterion_300,
        device
    )

    history_300["train_loss"].append(train_loss)
    history_300["train_accuracy"].append(train_accuracy)
    history_300["val_loss"].append(val_loss)
    history_300["val_accuracy"].append(val_accuracy)

    print(f"Train Loss:      {train_loss:.4f}")
    print(f"Train Accuracy:  {train_accuracy * 100:.2f}%")
    print(f"Val Loss:        {val_loss:.4f}")
    print(f"Val Accuracy:    {val_accuracy * 100:.2f}%")

    if val_loss < best_val_loss_300:

        best_val_loss_300 = val_loss

        best_model_state_300 = {
            key: value.cpu().clone()
            for key, value in model_300.state_dict().items()
        }

        print("✅ Best 300x300 model updated!")

# Restore best validation model
model_300.load_state_dict(best_model_state_300)
model_300 = model_300.to(device)

print("\n" + "=" * 50)
print("300x300 training complete!")
print(f"Best Validation Loss: {best_val_loss_300:.4f}")
print("=" * 50)

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

# Create same EfficientNet-B0 architecture
gradcam_model = efficientnet_b0(
    weights=None
)

# Same 5-class classifier
num_features = gradcam_model.classifier[1].in_features

gradcam_model.classifier[1] = nn.Linear(
    num_features,
    5
)

# Load final trained model
checkpoint = torch.load(
    "models/retina_xai_final_efficientnet_b0.pth",
    map_location=device
)

gradcam_model.load_state_dict(
    checkpoint["model_state_dict"]
)

gradcam_model = gradcam_model.to(device)
gradcam_model.eval()

print("Final model loaded successfully!")
print("Device:", device)
print("Classes:", checkpoint["num_classes"])
print(
    "Saved test accuracy:",
    checkpoint["test_accuracy"] * 100
)

In [ ]:
print(gradcam_model.features[-1])

In [ ]:
target_layer = gradcam_model.features[-1][0]

print("Target layer selected:")
print(target_layer)

In [ ]:
target_layer = gradcam_model.features[-1][0]

print("Grad-CAM target layer selected successfully!")
print(target_layer)

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

print("Grad-CAM imported successfully!")

In [ ]:
target_layer = gradcam_model.features[-1][0]

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import efficientnet_b0

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Create EfficientNet-B0
gradcam_model = efficientnet_b0(weights=None)

# Change classifier for 5 DR classes
num_features = gradcam_model.classifier[1].in_features
gradcam_model.classifier[1] = nn.Linear(num_features, 5)

# Load our trained model
MODEL_PATH = "models/retina_xai_final_efficientnet_b0.pth"

checkpoint = torch.load(
    MODEL_PATH,
    map_location=device
)

gradcam_model.load_state_dict(checkpoint["model_state_dict"])

# Move model to device
gradcam_model = gradcam_model.to(device)

# Evaluation mode
gradcam_model.eval()

print("Final model loaded successfully!")
print("Device:", device)
print("Classes:", checkpoint["num_classes"])
print("Saved test accuracy:", checkpoint["test_accuracy"])

In [ ]:
target_layer = gradcam_model.features[-1][0]

print("Grad-CAM target layer selected successfully!")
print(target_layer)

In [ ]:
from pytorch_grad_cam import GradCAM

cam = GradCAM(
    model=gradcam_model,
    target_layers=[target_layer]
)

print("Grad-CAM object created successfully!")

In [ ]:
import os
import glob

BASE_DIR = r"C:\Users\vu241\OneDrive\Desktop\Diabetic-Retinopathy-Screening\Disease Dataset\B. Disease Grading"

TRAIN_DIR = os.path.join(
    BASE_DIR,
    "1. Original Images",
    "a. Training Set"
)

TEST_DIR = os.path.join(
    BASE_DIR,
    "1. Original Images",
    "b. Testing Set"
)

GROUNDTRUTH_DIR = os.path.join(
    BASE_DIR,
    "2. Groundtruths"
)

print("TEST_DIR:", TEST_DIR)
print("Test directory exists:", os.path.exists(TEST_DIR))

In [ ]:
test_images = glob.glob(os.path.join(TEST_DIR, "*"))

print("Total test images:", len(test_images))
print("First image:", test_images[0])

In [ ]:
import cv2
import numpy as np

def preprocess_fundus(image_path, size=224):
    image = cv2.imread(image_path)

    # BGR → RGB
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Create mask to remove black background
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

    _, mask = cv2.threshold(
        gray,
        10,
        255,
        cv2.THRESH_BINARY
    )

    # Find largest contour
    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    if contours:
        largest_contour = max(
            contours,
            key=cv2.contourArea
        )

        x, y, w, h = cv2.boundingRect(largest_contour)

        image = image[y:y+h, x:x+w]

    # Resize
    image = cv2.resize(
        image,
        (size, size)
    )

    return image

print("Fundus preprocessing function restored!")

In [ ]:
from PIL import Image
import torch
from torchvision import transforms

image_path = test_images[0]

original_image = Image.open(image_path).convert("RGB")

image = preprocess_fundus(image_path)
image = Image.fromarray(image)

inference_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

input_tensor = inference_transform(image).unsqueeze(0).to(device)

print("Image:", os.path.basename(image_path))
print("Input shape:", input_tensor.shape)

In [ ]:
class_names = {
    0: "No DR",
    1: "Mild",
    2: "Moderate",
    3: "Severe",
    4: "Proliferative"
}

gradcam_model.eval()

with torch.no_grad():
    outputs = gradcam_model(input_tensor)
    probabilities = torch.softmax(outputs, dim=1)

predicted_class = torch.argmax(probabilities, dim=1).item()
confidence = probabilities[0, predicted_class].item()

print("Image:", os.path.basename(image_path))
print("Predicted Grade:", predicted_class)
print("Predicted Class:", class_names[predicted_class])
print(f"Confidence: {confidence * 100:.2f}%")

print("\nClass Probabilities:")
for i, prob in enumerate(probabilities[0]):
    print(f"{class_names[i]:15s}: {prob.item() * 100:.2f}%")

In [ ]:
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
import matplotlib.pyplot as plt
import numpy as np

# Target the class predicted by the model
targets = [ClassifierOutputTarget(predicted_class)]

# Generate Grad-CAM
grayscale_cam = cam(
    input_tensor=input_tensor,
    targets=targets
)

# Remove batch dimension
grayscale_cam = grayscale_cam[0]

# Prepare image for visualization
rgb_image = np.array(image.resize((224, 224))).astype(np.float32) / 255.0

# Overlay Grad-CAM heatmap on fundus image
visualization = show_cam_on_image(
    rgb_image,
    grayscale_cam,
    use_rgb=True
)

# Display
plt.figure(figsize=(8, 8))
plt.imshow(visualization)
plt.title(
    f"Grad-CAM — {class_names[predicted_class]} "
    f"({confidence * 100:.2f}%)"
)
plt.axis("off")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Original processed image
original_display = np.array(image.resize((224, 224)))

# Convert Grad-CAM to heatmap
import cv2

heatmap = cv2.applyColorMap(
    np.uint8(255 * grayscale_cam),
    cv2.COLORMAP_JET
)

heatmap = cv2.cvtColor(
    heatmap,
    cv2.COLOR_BGR2RGB
)

# Create overlay
overlay = show_cam_on_image(
    rgb_image,
    grayscale_cam,
    use_rgb=True
)

# Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Original
axes[0].imshow(original_display)
axes[0].set_title("Original Fundus Image", fontsize=14)
axes[0].axis("off")

# Heatmap
axes[1].imshow(heatmap)
axes[1].set_title("Grad-CAM Heatmap", fontsize=14)
axes[1].axis("off")

# Overlay
axes[2].imshow(overlay)
axes[2].set_title(
    f"Grad-CAM Overlay\n"
    f"{class_names[predicted_class]} — {confidence * 100:.2f}%",
    fontsize=14
)
axes[2].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
def generate_gradcam(image_path):
    """
    Predict DR grade and generate Grad-CAM visualization
    for a single fundus image.
    """

    # -----------------------------
    # 1. Preprocess image
    # -----------------------------
    processed = preprocess_fundus(image_path)

    processed_pil = Image.fromarray(processed)

    # -----------------------------
    # 2. Transform for model
    # -----------------------------
    input_tensor = inference_transform(
        processed_pil
    ).unsqueeze(0).to(device)

    # -----------------------------
    # 3. Model prediction
    # -----------------------------
    gradcam_model.eval()

    with torch.no_grad():
        outputs = gradcam_model(input_tensor)
        probabilities = torch.softmax(outputs, dim=1)

    predicted_class = torch.argmax(
        probabilities,
        dim=1
    ).item()

    confidence = probabilities[
        0, predicted_class
    ].item()

    # -----------------------------
    # 4. Generate Grad-CAM
    # -----------------------------
    targets = [
        ClassifierOutputTarget(predicted_class)
    ]

    grayscale_cam = cam(
        input_tensor=input_tensor,
        targets=targets
    )[0]

    # -----------------------------
    # 5. Prepare visualization
    # -----------------------------
    rgb_image = np.array(
        processed_pil.resize((224, 224))
    ).astype(np.float32) / 255.0

    heatmap = cv2.applyColorMap(
        np.uint8(255 * grayscale_cam),
        cv2.COLORMAP_JET
    )

    heatmap = cv2.cvtColor(
        heatmap,
        cv2.COLOR_BGR2RGB
    )

    overlay = show_cam_on_image(
        rgb_image,
        grayscale_cam,
        use_rgb=True
    )

    # -----------------------------
    # 6. Display results
    # -----------------------------
    fig, axes = plt.subplots(
        1, 3,
        figsize=(18, 6)
    )

    # Original
    axes[0].imshow(rgb_image)
    axes[0].set_title(
        "Original Fundus Image",
        fontsize=14
    )
    axes[0].axis("off")

    # Heatmap
    axes[1].imshow(heatmap)
    axes[1].set_title(
        "Grad-CAM Heatmap",
        fontsize=14
    )
    axes[1].axis("off")

    # Overlay
    axes[2].imshow(overlay)
    axes[2].set_title(
        f"Grad-CAM Overlay\n"
        f"{class_names[predicted_class]} "
        f"— {confidence * 100:.2f}%",
        fontsize=14
    )
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()

    # -----------------------------
    # 7. Return result
    # -----------------------------
    return {
        "image": os.path.basename(image_path),
        "predicted_grade": predicted_class,
        "predicted_class": class_names[predicted_class],
        "confidence": confidence,
        "heatmap": heatmap,
        "overlay": overlay
    }

In [ ]:
result = generate_gradcam(test_images[0])

In [ ]:
print("Prediction:", result["predicted_class"])
print(f"Confidence: {result['confidence'] * 100:.2f}%")

In [ ]:
# Test Grad-CAM on another retinal image

test_image_2 = test_images[20]

result_2 = generate_gradcam(test_image_2)

print("Image:", result_2["image"])
print("Prediction:", result_2["predicted_class"])
print(f"Confidence: {result_2['confidence'] * 100:.2f}%")

In [ ]:
def predict_retina(image_path, review_threshold=0.60):
    """
    Complete RETINA-XAI inference pipeline.

    Returns:
        predicted DR grade
        class name
        confidence
        risk level
        doctor review recommendation
    """

    # -----------------------------
    # 1. Preprocess image
    # -----------------------------
    processed = preprocess_fundus(image_path)
    processed_pil = Image.fromarray(processed)

    # -----------------------------
    # 2. Transform
    # -----------------------------
    input_tensor = inference_transform(
        processed_pil
    ).unsqueeze(0).to(device)

    # -----------------------------
    # 3. Prediction
    # -----------------------------
    gradcam_model.eval()

    with torch.no_grad():
        outputs = gradcam_model(input_tensor)
        probabilities = torch.softmax(outputs, dim=1)

    predicted_class = torch.argmax(
        probabilities, dim=1
    ).item()

    confidence = probabilities[
        0, predicted_class
    ].item()

    # -----------------------------
    # 4. Risk level
    # -----------------------------
    if predicted_class == 0:
        risk_level = "Low"
    elif predicted_class == 1:
        risk_level = "Low-Mild"
    elif predicted_class == 2:
        risk_level = "Moderate"
    elif predicted_class == 3:
        risk_level = "High"
    else:
        risk_level = "Very High"

    # -----------------------------
    # 5. Doctor review
    # -----------------------------
    doctor_review_required = (
        confidence < review_threshold
        or predicted_class >= 2
    )

    # -----------------------------
    # 6. Return result
    # -----------------------------
    return {
        "image": os.path.basename(image_path),
        "predicted_grade": predicted_class,
        "predicted_class": class_names[predicted_class],
        "confidence": confidence,
        "risk_level": risk_level,
        "doctor_review_required": doctor_review_required
    }

In [ ]:
result = predict_retina(test_images[0])

print("===================================")
print("       RETINA-XAI AI RESULT")
print("===================================")

print("Image:", result["image"])
print("DR Grade:", result["predicted_grade"])
print("Class:", result["predicted_class"])
print(f"Confidence: {result['confidence'] * 100:.2f}%")
print("Risk Level:", result["risk_level"])
print("Doctor Review:", result["doctor_review_required"])

In [ ]:
def explain_prediction(image_path, review_threshold=0.60):
    """
    Complete RETINA-XAI prediction + Grad-CAM pipeline.
    """

    # -----------------------------
    # 1. Preprocess image
    # -----------------------------
    processed = preprocess_fundus(image_path)
    processed_pil = Image.fromarray(processed)

    # -----------------------------
    # 2. Prepare model input
    # -----------------------------
    input_tensor = inference_transform(
        processed_pil
    ).unsqueeze(0).to(device)

    # -----------------------------
    # 3. Model prediction
    # -----------------------------
    gradcam_model.eval()

    with torch.no_grad():
        outputs = gradcam_model(input_tensor)
        probabilities = torch.softmax(outputs, dim=1)

    predicted_class = torch.argmax(
        probabilities,
        dim=1
    ).item()

    confidence = probabilities[
        0, predicted_class
    ].item()

    # -----------------------------
    # 4. Risk level
    # -----------------------------
    if predicted_class == 0:
        risk_level = "Low"
    elif predicted_class == 1:
        risk_level = "Low-Mild"
    elif predicted_class == 2:
        risk_level = "Moderate"
    elif predicted_class == 3:
        risk_level = "High"
    else:
        risk_level = "Very High"

    # -----------------------------
    # 5. Doctor review
    # -----------------------------
    doctor_review_required = (
        confidence < review_threshold
        or predicted_class >= 2
    )

    # -----------------------------
    # 6. Grad-CAM
    # -----------------------------
    targets = [
        ClassifierOutputTarget(predicted_class)
    ]

    grayscale_cam = cam(
        input_tensor=input_tensor,
        targets=targets
    )[0]

    # -----------------------------
    # 7. Prepare visualization
    # -----------------------------
    rgb_image = np.array(
        processed_pil.resize((224, 224))
    ).astype(np.float32) / 255.0

    heatmap = cv2.applyColorMap(
        np.uint8(255 * grayscale_cam),
        cv2.COLORMAP_JET
    )

    heatmap = cv2.cvtColor(
        heatmap,
        cv2.COLOR_BGR2RGB
    )

    overlay = show_cam_on_image(
        rgb_image,
        grayscale_cam,
        use_rgb=True
    )

    # -----------------------------
    # 8. Display
    # -----------------------------
    fig, axes = plt.subplots(
        1, 3,
        figsize=(18, 6)
    )

    axes[0].imshow(rgb_image)
    axes[0].set_title(
        "Original Fundus",
        fontsize=14
    )
    axes[0].axis("off")

    axes[1].imshow(heatmap)
    axes[1].set_title(
        "Grad-CAM Heatmap",
        fontsize=14
    )
    axes[1].axis("off")

    axes[2].imshow(overlay)
    axes[2].set_title(
        f"{class_names[predicted_class]}\n"
        f"Confidence: {confidence * 100:.2f}%",
        fontsize=14
    )
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()

    # -----------------------------
    # 9. Return complete result
    # -----------------------------
    return {
        "image": os.path.basename(image_path),
        "predicted_grade": predicted_class,
        "predicted_class": class_names[predicted_class],
        "confidence": confidence,
        "risk_level": risk_level,
        "doctor_review_required": doctor_review_required,
        "heatmap": heatmap,
        "overlay": overlay
    }

In [ ]:
final_result = explain_prediction(test_images[0])

print("===================================")
print("       RETINA-XAI FINAL RESULT")
print("===================================")

print("Image:", final_result["image"])
print("DR Grade:", final_result["predicted_grade"])
print("Class:", final_result["predicted_class"])
print(f"Confidence: {final_result['confidence'] * 100:.2f}%")
print("Risk Level:", final_result["risk_level"])
print("Doctor Review:", final_result["doctor_review_required"])

In [ ]:
import os

GRADCAM_DIR = os.path.join(
    "outputs",
    "gradcam"
)

os.makedirs(GRADCAM_DIR, exist_ok=True)

print("Grad-CAM output directory ready:")
print(GRADCAM_DIR)

In [ ]:
image_name = os.path.splitext(
    final_result["image"]
)[0]

original_path = os.path.join(
    GRADCAM_DIR,
    f"{image_name}_original.png"
)

heatmap_path = os.path.join(
    GRADCAM_DIR,
    f"{image_name}_heatmap.png"
)

overlay_path = os.path.join(
    GRADCAM_DIR,
    f"{image_name}_overlay.png"
)

# Save original
Image.fromarray(
    (rgb_image * 255).astype(np.uint8)
).save(original_path)

# Save heatmap
Image.fromarray(
    heatmap
).save(heatmap_path)

# Save overlay
Image.fromarray(
    overlay
).save(overlay_path)

print("Saved successfully!")
print(original_path)
print(heatmap_path)
print(overlay_path)

In [ ]:
print("Original exists:", os.path.exists(original_path))
print("Heatmap exists:", os.path.exists(heatmap_path))
print("Overlay exists:", os.path.exists(overlay_path))

In [ ]:
import pandas as pd

TEST_LABEL_FILE = os.path.join(
    GROUNDTRUTH_DIR,
    "b. IDRiD_Disease Grading_Testing Labels.csv"
)

test_df = pd.read_csv(TEST_LABEL_FILE)

test_df.columns = test_df.columns.str.strip()

print(test_df.head())
print("\nTotal testing labels:", len(test_df))

In [ ]:
# Images we tested
selected_images = [
    "IDRiD_001",
    "IDRiD_021",
    "IDRiD_041",
    "IDRiD_061",
    "IDRiD_081"
]

# Predictions obtained from our tests
predictions = {
    "IDRiD_001": 4,
    "IDRiD_021": 3,
    "IDRiD_041": 0,
    "IDRiD_061": 4,
    "IDRiD_081": 0
}

# Get actual labels
comparison = test_df[
    test_df["Image name"].isin(selected_images)
][["Image name", "Retinopathy grade"]].copy()

# Add predictions
comparison["Predicted grade"] = comparison["Image name"].map(
    predictions
)

# Check correctness
comparison["Correct"] = (
    comparison["Retinopathy grade"]
    == comparison["Predicted grade"]
)

# Sort in our selected order
comparison["order"] = comparison["Image name"].apply(
    selected_images.index
)

comparison = comparison.sort_values("order").drop(
    columns="order"
)

print(comparison.to_string(index=False))

In [2]:
# APTOS

In [3]:
import os
import pandas as pd

APTOS_DIR = r"C:\Users\vu241\OneDrive\Desktop\Diabetic-Retinopathy-Screening\Disease Dataset\APTOS 2019"

APTOS_TRAIN_DIR = os.path.join(
    APTOS_DIR,
    "train_images"
)

APTOS_TEST_DIR = os.path.join(
    APTOS_DIR,
    "test_images"
)

APTOS_TRAIN_CSV = os.path.join(
    APTOS_DIR,
    "train.csv"
)

APTOS_TEST_CSV = os.path.join(
    APTOS_DIR,
    "test.csv"
)

print("APTOS directory exists:", os.path.exists(APTOS_DIR))
print("Train images exists:", os.path.exists(APTOS_TRAIN_DIR))
print("Test images exists:", os.path.exists(APTOS_TEST_DIR))
print("Train CSV exists:", os.path.exists(APTOS_TRAIN_CSV))
print("Test CSV exists:", os.path.exists(APTOS_TEST_CSV))

APTOS directory exists: True
Train images exists: True
Test images exists: True
Train CSV exists: True
Test CSV exists: True


In [4]:
aptos_df = pd.read_csv(APTOS_TRAIN_CSV)

print("Total training rows:", len(aptos_df))

print("\nColumns:")
print(aptos_df.columns.tolist())

print("\nFirst 5 rows:")
display(aptos_df.head())

print("\nClass distribution:")
print(
    aptos_df["diagnosis"]
    .value_counts()
    .sort_index()
)

Total training rows: 3662

Columns:
['id_code', 'diagnosis']

First 5 rows:


,id_code,diagnosis
0,000c1434d8d7,2
1,001639a390f0,4
2,0024cdab0c1e,1
3,002c21358ce6,0
4,005b95c28852,0



Class distribution:
diagnosis
0    1805
1     370
2     999
3     193
4     295
Name: count, dtype: int64


In [5]:
# Create full image paths for APTOS

def find_aptos_image(image_id):
    """
    Find the actual APTOS image file.
    APTOS images are usually stored as .png files.
    """
    
    image_path = os.path.join(
        APTOS_TRAIN_DIR,
        image_id + ".png"
    )
    
    if os.path.exists(image_path):
        return image_path
    
    return None


aptos_df["image_path"] = aptos_df["id_code"].apply(
    find_aptos_image
)

print("Total rows:", len(aptos_df))
print("Missing images:", aptos_df["image_path"].isna().sum())

Total rows: 3662
Missing images: 0


In [6]:
aptos_df["label"] = aptos_df["diagnosis"].astype(int)

aptos_df = aptos_df[
    ["id_code", "image_path", "label"]
].copy()

print(aptos_df.head())
print("\nTotal images:", len(aptos_df))

        id_code                                         image_path  label
0  000c1434d8d7  C:\Users\vu241\OneDrive\Desktop\Diabetic-Retin...      2
1  001639a390f0  C:\Users\vu241\OneDrive\Desktop\Diabetic-Retin...      4
2  0024cdab0c1e  C:\Users\vu241\OneDrive\Desktop\Diabetic-Retin...      1
3  002c21358ce6  C:\Users\vu241\OneDrive\Desktop\Diabetic-Retin...      0
4  005b95c28852  C:\Users\vu241\OneDrive\Desktop\Diabetic-Retin...      0

Total images: 3662


In [7]:
from sklearn.model_selection import train_test_split

aptos_train_df, aptos_val_df = train_test_split(
    aptos_df,
    test_size=0.20,
    random_state=42,
    stratify=aptos_df["label"]
)

print("APTOS training images:", len(aptos_train_df))
print("APTOS validation images:", len(aptos_val_df))

print("\nTraining distribution:")
print(
    aptos_train_df["label"]
    .value_counts()
    .sort_index()
)

print("\nValidation distribution:")
print(
    aptos_val_df["label"]
    .value_counts()
    .sort_index()
)

APTOS training images: 2929
APTOS validation images: 733

Training distribution:
label
0    1444
1     296
2     799
3     154
4     236
Name: count, dtype: int64

Validation distribution:
label
0    361
1     74
2    200
3     39
4     59
Name: count, dtype: int64


In [8]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image

print("Dataset and DataLoader imported successfully!")

Dataset and DataLoader imported successfully!


In [9]:
class APTOSRetinaDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        image_path = self.dataframe.loc[index, "image_path"]
        label = self.dataframe.loc[index, "label"]

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = torch.tensor(label, dtype=torch.long)

        return image, label


print("APTOS Dataset class created successfully!")

APTOS Dataset class created successfully!


In [10]:
from torchvision import transforms

print("transforms imported successfully!")

transforms imported successfully!


In [11]:
aptos_train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

aptos_val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("APTOS transforms created successfully!")

APTOS transforms created successfully!


In [12]:
aptos_train_dataset = APTOSRetinaDataset(
    aptos_train_df,
    transform=aptos_train_transform
)

aptos_val_dataset = APTOSRetinaDataset(
    aptos_val_df,
    transform=aptos_val_transform
)

print("APTOS training dataset:", len(aptos_train_dataset))
print("APTOS validation dataset:", len(aptos_val_dataset))

APTOS training dataset: 2929
APTOS validation dataset: 733


In [13]:
aptos_train_loader = DataLoader(
    aptos_train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0
)

aptos_val_loader = DataLoader(
    aptos_val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0
)

print("APTOS DataLoaders created successfully!")

APTOS DataLoaders created successfully!


In [14]:
import torch

print("PyTorch imported successfully!")
print("PyTorch version:", torch.__version__)

PyTorch imported successfully!
PyTorch version: 2.14.0+cpu


In [15]:
images, labels = next(iter(aptos_train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Labels:", labels)

Image batch shape: torch.Size([16, 3, 224, 224])
Label batch shape: torch.Size([16])
Labels: tensor([2, 2, 0, 0, 0, 0, 4, 1, 0, 1, 0, 0, 2, 0, 0, 0])


In [16]:
from torch.utils.data import WeightedRandomSampler

# Count samples in each class
class_counts = aptos_train_df["label"].value_counts().sort_index()

print("Training class counts:")
print(class_counts)

# Give higher weight to minority classes
class_sample_weights = 1.0 / class_counts.values

print("\nClass weights:")
print(class_sample_weights)

# Assign weight to every training image
sample_weights = aptos_train_df["label"].map(
    lambda label: class_sample_weights[label]
).values

sample_weights = torch.tensor(
    sample_weights,
    dtype=torch.double
)

# Create balanced sampler
aptos_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

print("\nBalanced sampler created successfully!")

Training class counts:
label
0    1444
1     296
2     799
3     154
4     236
Name: count, dtype: int64

Class weights:
[0.00069252 0.00337838 0.00125156 0.00649351 0.00423729]

Balanced sampler created successfully!


In [17]:
aptos_train_loader_balanced = DataLoader(
    aptos_train_dataset,
    batch_size=16,
    sampler=aptos_sampler,
    num_workers=0
)

print("Balanced APTOS DataLoader created successfully!")

Balanced APTOS DataLoader created successfully!


In [18]:
images, labels = next(iter(aptos_train_loader_balanced))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Labels:", labels)

Image batch shape: torch.Size([16, 3, 224, 224])
Label batch shape: torch.Size([16])
Labels: tensor([3, 3, 1, 0, 4, 4, 1, 4, 1, 3, 2, 4, 1, 2, 0, 4])


In [19]:
import torch
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

# Load pretrained EfficientNet-B0
weights = EfficientNet_B0_Weights.DEFAULT

aptos_model = efficientnet_b0(weights=weights)

# Replace final classifier
num_features = aptos_model.classifier[1].in_features

aptos_model.classifier[1] = nn.Linear(
    num_features,
    5
)

# Move model to device
aptos_model = aptos_model.to(device)

print("EfficientNet-B0 created successfully!")
print("Number of input features:", num_features)
print("Number of output classes:", 5)

Device: cpu
EfficientNet-B0 created successfully!
Number of input features: 1280
Number of output classes: 5


In [20]:
print(34)

34


In [21]:
# Class counts from APTOS training data
class_counts = aptos_train_df["label"].value_counts().sort_index()

# Calculate inverse-frequency class weights
class_weights = 1.0 / torch.tensor(
    class_counts.values,
    dtype=torch.float32
)

# Normalize weights so their average is 1
class_weights = class_weights / class_weights.mean()

# Move weights to CPU/GPU along with the model
class_weights = class_weights.to(device)

print("Class weights:")
print(class_weights)

# Weighted Cross Entropy Loss
criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

# Adam optimizer
optimizer = torch.optim.Adam(
    aptos_model.parameters(),
    lr=1e-4
)

print("\nLoss function and optimizer created successfully!")

Class weights:
tensor([0.2157, 1.0522, 0.3898, 2.0225, 1.3198])

Loss function and optimizer created successfully!


In [22]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    total_batches = len(loader)

    for batch_idx, (images, labels) in enumerate(loader, start=1):

        images = images.to(device)
        labels = labels.to(device)

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)

        # Calculate loss
        loss = criterion(outputs, labels)

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        # Statistics
        running_loss += loss.item() * images.size(0)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        # Show progress every 20 batches
        if batch_idx % 20 == 0 or batch_idx == total_batches:
            current_acc = 100.0 * correct / total

            print(
                f"Batch {batch_idx}/{total_batches} | "
                f"Loss: {loss.item():.4f} | "
                f"Accuracy: {current_acc:.2f}%"
            )

    epoch_loss = running_loss / total
    epoch_accuracy = 100.0 * correct / total

    return epoch_loss, epoch_accuracy


def validate_one_epoch(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_accuracy = 100.0 * correct / total

    return epoch_loss, epoch_accuracy


print("Training and validation functions updated successfully!")

Training and validation functions updated successfully!


In [23]:
train_loss, train_acc = train_one_epoch(
    aptos_model,
    aptos_train_loader_balanced,
    criterion,
    optimizer,
    device
)

print("\nTraining finished!")
print(f"Train Loss: {train_loss:.4f}")
print(f"Train Accuracy: {train_acc:.2f}%")

Batch 20/184 | Loss: 1.4143 | Accuracy: 26.88%
Batch 40/184 | Loss: 1.3423 | Accuracy: 32.03%
Batch 60/184 | Loss: 1.1155 | Accuracy: 36.46%
Batch 80/184 | Loss: 1.1235 | Accuracy: 38.59%
Batch 100/184 | Loss: 1.1145 | Accuracy: 41.75%
Batch 120/184 | Loss: 0.9095 | Accuracy: 43.54%
Batch 140/184 | Loss: 0.5896 | Accuracy: 45.62%
Batch 160/184 | Loss: 0.8597 | Accuracy: 47.03%
Batch 180/184 | Loss: 0.5320 | Accuracy: 48.85%
Batch 184/184 | Loss: 1.9091 | Accuracy: 49.10%

Training finished!
Train Loss: 1.0632
Train Accuracy: 49.10%


In [24]:
val_loss, val_acc = validate_one_epoch(
    aptos_model,
    aptos_val_loader,
    criterion,
    device
)

print("\n========== EPOCH 1 VALIDATION ==========")
print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_acc:.2f}%")


========== EPOCH 1 VALIDATION ==========
Validation Loss: 1.1647
Validation Accuracy: 56.34%


In [25]:
train_loss, train_acc = train_one_epoch(
    aptos_model,
    aptos_train_loader_balanced,
    criterion,
    optimizer,
    device
)

print("\n========== EPOCH 2 TRAINING COMPLETE ==========")
print(f"Train Loss: {train_loss:.4f}")
print(f"Train Accuracy: {train_acc:.2f}%")

Batch 20/184 | Loss: 1.2078 | Accuracy: 61.88%
Batch 40/184 | Loss: 1.0819 | Accuracy: 61.25%
Batch 60/184 | Loss: 0.7767 | Accuracy: 61.15%
Batch 80/184 | Loss: 0.4580 | Accuracy: 62.50%
Batch 100/184 | Loss: 0.3879 | Accuracy: 62.31%
Batch 120/184 | Loss: 0.8448 | Accuracy: 62.40%
Batch 140/184 | Loss: 0.5720 | Accuracy: 63.39%
Batch 160/184 | Loss: 0.6096 | Accuracy: 64.10%
Batch 180/184 | Loss: 0.3580 | Accuracy: 64.55%
Batch 184/184 | Loss: 1.2621 | Accuracy: 64.80%

========== EPOCH 2 TRAINING COMPLETE ==========
Train Loss: 0.6845
Train Accuracy: 64.80%


In [26]:
val_loss, val_acc = validate_one_epoch(
    aptos_model,
    aptos_val_loader,
    criterion,
    device
)

print("\n========== EPOCH 2 VALIDATION ==========")
print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_acc:.2f}%")


========== EPOCH 2 VALIDATION ==========
Validation Loss: 1.0682
Validation Accuracy: 64.53%


In [27]:
train_loss, train_acc = train_one_epoch(
    aptos_model,
    aptos_train_loader_balanced,
    criterion,
    optimizer,
    device
)

print("\n========== EPOCH 3 TRAINING COMPLETE ==========")
print(f"Train Loss: {train_loss:.4f}")
print(f"Train Accuracy: {train_acc:.2f}%")

Batch 20/184 | Loss: 0.3448 | Accuracy: 70.94%
Batch 40/184 | Loss: 0.8328 | Accuracy: 71.56%
Batch 60/184 | Loss: 0.4973 | Accuracy: 70.62%
Batch 80/184 | Loss: 0.2945 | Accuracy: 71.33%
Batch 100/184 | Loss: 0.3657 | Accuracy: 70.62%
Batch 120/184 | Loss: 0.4294 | Accuracy: 70.21%
Batch 140/184 | Loss: 0.5058 | Accuracy: 70.85%
Batch 160/184 | Loss: 0.6245 | Accuracy: 71.29%
Batch 180/184 | Loss: 0.5041 | Accuracy: 71.46%
Batch 184/184 | Loss: 4.0885 | Accuracy: 71.39%

========== EPOCH 3 TRAINING COMPLETE ==========
Train Loss: 0.4993
Train Accuracy: 71.39%


In [28]:
val_loss, val_acc = validate_one_epoch(
    aptos_model,
    aptos_val_loader,
    criterion,
    device
)

print("\n========== EPOCH 3 VALIDATION ==========")
print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_acc:.2f}%")


========== EPOCH 3 VALIDATION ==========
Validation Loss: 1.0990
Validation Accuracy: 65.48%


In [29]:
train_loss, train_acc = train_one_epoch(
    aptos_model,
    aptos_train_loader_balanced,
    criterion,
    optimizer,
    device
)

print("\n========== EPOCH 4 TRAINING COMPLETE ==========")
print(f"Train Loss: {train_loss:.4f}")
print(f"Train Accuracy: {train_acc:.2f}%")

Batch 20/184 | Loss: 0.2819 | Accuracy: 75.62%
Batch 40/184 | Loss: 0.5694 | Accuracy: 74.84%
Batch 60/184 | Loss: 0.5043 | Accuracy: 74.06%
Batch 80/184 | Loss: 0.3213 | Accuracy: 74.38%
Batch 100/184 | Loss: 0.7131 | Accuracy: 73.88%
Batch 120/184 | Loss: 0.4469 | Accuracy: 74.17%
Batch 140/184 | Loss: 0.3034 | Accuracy: 75.40%
Batch 160/184 | Loss: 0.3252 | Accuracy: 75.55%
Batch 180/184 | Loss: 0.2668 | Accuracy: 76.01%
Batch 184/184 | Loss: 1.3625 | Accuracy: 76.07%

========== EPOCH 4 TRAINING COMPLETE ==========
Train Loss: 0.4072
Train Accuracy: 76.07%


In [30]:
val_loss, val_acc = validate_one_epoch(
    aptos_model,
    aptos_val_loader,
    criterion,
    device
)

print("\n========== EPOCH 4 VALIDATION ==========")
print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_acc:.2f}%")


========== EPOCH 4 VALIDATION ==========
Validation Loss: 1.0083
Validation Accuracy: 70.40%


In [31]:
train_loss, train_acc = train_one_epoch(
    aptos_model,
    aptos_train_loader_balanced,
    criterion,
    optimizer,
    device
)

print("\n========== EPOCH 5 TRAINING COMPLETE ==========")
print(f"Train Loss: {train_loss:.4f}")
print(f"Train Accuracy: {train_acc:.2f}%")

Batch 20/184 | Loss: 0.2873 | Accuracy: 81.25%
Batch 40/184 | Loss: 0.3654 | Accuracy: 79.53%
Batch 60/184 | Loss: 0.4346 | Accuracy: 78.75%
Batch 80/184 | Loss: 0.5499 | Accuracy: 78.91%
Batch 100/184 | Loss: 0.2605 | Accuracy: 79.31%
Batch 120/184 | Loss: 0.2022 | Accuracy: 79.95%
Batch 140/184 | Loss: 0.1850 | Accuracy: 79.42%
Batch 160/184 | Loss: 0.5781 | Accuracy: 79.80%
Batch 180/184 | Loss: 0.4170 | Accuracy: 79.93%
Batch 184/184 | Loss: 1.6530 | Accuracy: 80.03%

========== EPOCH 5 TRAINING COMPLETE ==========
Train Loss: 0.3329
Train Accuracy: 80.03%


In [32]:
val_loss, val_acc = validate_one_epoch(
    aptos_model,
    aptos_val_loader,
    criterion,
    device
)

print("\n========== EPOCH 5 VALIDATION ==========")
print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_acc:.2f}%")


========== EPOCH 5 VALIDATION ==========
Validation Loss: 1.1201
Validation Accuracy: 71.08%


In [33]:
import os
import torch

APTOS_MODEL_PATH = os.path.join(
    "models",
    "retina_xai_aptos_efficientnet_b0.pth"
)

os.makedirs("models", exist_ok=True)

torch.save({
    "model_state_dict": aptos_model.state_dict(),
    "num_classes": 5,
    "class_names": {
        0: "No DR",
        1: "Mild",
        2: "Moderate",
        3: "Severe",
        4: "Proliferative"
    },
    "epoch": 5,
    "val_accuracy": 71.08,
    "val_loss": 1.1201
}, APTOS_MODEL_PATH)

print("APTOS model saved successfully!")
print("Path:", APTOS_MODEL_PATH)

APTOS model saved successfully!
Path: models\retina_xai_aptos_efficientnet_b0.pth


In [34]:
import torch
import torch.nn as nn
from torchvision.models import efficientnet_b0

# Create a fresh EfficientNet-B0
transfer_model = efficientnet_b0(weights=None)

# Replace classifier with 5 DR classes
num_features = transfer_model.classifier[1].in_features

transfer_model.classifier[1] = nn.Linear(
    num_features,
    5
)

# Load APTOS-trained weights
checkpoint = torch.load(
    "models/retina_xai_aptos_efficientnet_b0.pth",
    map_location=device
)

transfer_model.load_state_dict(
    checkpoint["model_state_dict"]
)

transfer_model = transfer_model.to(device)

print("APTOS-trained model loaded successfully!")
print("Starting validation accuracy:", checkpoint["val_accuracy"])
print("Starting validation loss:", checkpoint["val_loss"])
print("Device:", device)

APTOS-trained model loaded successfully!
Starting validation accuracy: 71.08
Starting validation loss: 1.1201
Device: cpu


In [37]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

# IDRiD paths
BASE_DIR = r"C:\Users\vu241\OneDrive\Desktop\Diabetic-Retinopathy-Screening\Disease Dataset\B. Disease Grading"

TRAIN_DIR = os.path.join(
    BASE_DIR,
    "1. Original Images",
    "a. Training Set"
)

GROUNDTRUTH_DIR = os.path.join(
    BASE_DIR,
    "2. Groundtruths"
)

TRAIN_LABEL_FILE = os.path.join(
    GROUNDTRUTH_DIR,
    "a. IDRiD_Disease Grading_Training Labels.csv"
)

# Load labels
df = pd.read_csv(TRAIN_LABEL_FILE)

# Clean column names
df.columns = df.columns.str.strip()

# Create image paths
def find_idrid_image(image_name):
    image_path = os.path.join(
        TRAIN_DIR,
        image_name + ".jpg"
    )

    if os.path.exists(image_path):
        return image_path

    return None

df["image_path"] = df["Image name"].apply(find_idrid_image)

# Create integer labels
df["label"] = df["Retinopathy grade"].astype(int)

# Keep required columns
idrid_df = df[
    ["Image name", "image_path", "label"]
].copy()

print("Total IDRiD images:", len(idrid_df))
print("Missing images:", idrid_df["image_path"].isna().sum())
print("\nClass distribution:")
print(idrid_df["label"].value_counts().sort_index())

Total IDRiD images: 413
Missing images: 0

Class distribution:
label
0    134
1     20
2    136
3     74
4     49
Name: count, dtype: int64


In [38]:
from sklearn.model_selection import train_test_split

# IDRiD training dataframe
idrid_df = df.copy()

# Make sure label is integer
idrid_df["label"] = idrid_df["Retinopathy grade"].astype(int)

# Keep required columns
idrid_df = idrid_df[
    ["Image name", "image_path", "label"]
].copy()

# Train/validation split
train_df, val_df = train_test_split(
    idrid_df,
    test_size=0.20,
    random_state=42,
    stratify=idrid_df["label"]
)

print("IDRiD Training images:", len(train_df))
print("IDRiD Validation images:", len(val_df))

print("\nIDRiD Training distribution:")
print(train_df["label"].value_counts().sort_index())

print("\nIDRiD Validation distribution:")
print(val_df["label"].value_counts().sort_index())

IDRiD Training images: 330
IDRiD Validation images: 83

IDRiD Training distribution:
label
0    107
1     16
2    109
3     59
4     39
Name: count, dtype: int64

IDRiD Validation distribution:
label
0    27
1     4
2    27
3    15
4    10
Name: count, dtype: int64


In [39]:
print("IDRiD Training images:", len(train_df))
print("IDRiD Validation images:", len(val_df))

print("\nIDRiD Training distribution:")
print(train_df["label"].value_counts().sort_index())

print("\nIDRiD Validation distribution:")
print(val_df["label"].value_counts().sort_index())

IDRiD Training images: 330
IDRiD Validation images: 83

IDRiD Training distribution:
label
0    107
1     16
2    109
3     59
4     39
Name: count, dtype: int64

IDRiD Validation distribution:
label
0    27
1     4
2    27
3    15
4    10
Name: count, dtype: int64


In [43]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("IDRiD transforms restored successfully!")

IDRiD transforms restored successfully!


In [44]:
idrid_train_dataset = PreprocessedRetinaDataset(
    train_df,
    transform=train_transform
)

idrid_val_dataset = PreprocessedRetinaDataset(
    val_df,
    transform=val_transform
)

print("IDRiD training dataset:", len(idrid_train_dataset))
print("IDRiD validation dataset:", len(idrid_val_dataset))

IDRiD training dataset: 330
IDRiD validation dataset: 83


In [41]:
import cv2
import torch
from torch.utils.data import Dataset
from PIL import Image


def preprocess_fundus(image_path, size=224):
    image = cv2.imread(image_path)

    image = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2RGB
    )

    # Find the fundus region
    gray = cv2.cvtColor(
        image,
        cv2.COLOR_RGB2GRAY
    )

    _, mask = cv2.threshold(
        gray,
        10,
        255,
        cv2.THRESH_BINARY
    )

    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    if contours:
        largest_contour = max(
            contours,
            key=cv2.contourArea
        )

        x, y, w, h = cv2.boundingRect(
            largest_contour
        )

        image = image[
            y:y+h,
            x:x+w
        ]

    image = cv2.resize(
        image,
        (size, size)
    )

    return image


class PreprocessedRetinaDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):

        image_path = self.dataframe.loc[
            index,
            "image_path"
        ]

        label = self.dataframe.loc[
            index,
            "label"
        ]

        image = preprocess_fundus(
            image_path
        )

        image = Image.fromarray(image)

        if self.transform:
            image = self.transform(image)

        label = torch.tensor(
            label,
            dtype=torch.long
        )

        return image, label


print("PreprocessedRetinaDataset restored successfully!")

PreprocessedRetinaDataset restored successfully!


In [45]:
from torch.utils.data import WeightedRandomSampler, DataLoader

# Count IDRiD training samples per class
idrid_class_counts = train_df["label"].value_counts().sort_index()

print("IDRiD training class counts:")
print(idrid_class_counts)

# Inverse-frequency weights
idrid_class_weights = 1.0 / idrid_class_counts.values

print("\nClass weights:")
print(idrid_class_weights)

# Weight for every training image
idrid_sample_weights = train_df["label"].map(
    lambda label: idrid_class_weights[label]
).values

idrid_sample_weights = torch.tensor(
    idrid_sample_weights,
    dtype=torch.double
)

# Balanced sampler
idrid_sampler = WeightedRandomSampler(
    weights=idrid_sample_weights,
    num_samples=len(idrid_sample_weights),
    replacement=True
)

print("\nIDRiD balanced sampler created successfully!")

IDRiD training class counts:
label
0    107
1     16
2    109
3     59
4     39
Name: count, dtype: int64

Class weights:
[0.00934579 0.0625     0.00917431 0.01694915 0.02564103]

IDRiD balanced sampler created successfully!


In [46]:
idrid_train_loader_balanced = DataLoader(
    idrid_train_dataset,
    batch_size=16,
    sampler=idrid_sampler,
    num_workers=0
)

idrid_val_loader = DataLoader(
    idrid_val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0
)

print("IDRiD transfer-learning DataLoaders created successfully!")
print("Training batches:", len(idrid_train_loader_balanced))
print("Validation batches:", len(idrid_val_loader))

IDRiD transfer-learning DataLoaders created successfully!
Training batches: 21
Validation batches: 6


In [47]:
images, labels = next(iter(idrid_train_loader_balanced))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Labels:", labels)

Image batch shape: torch.Size([16, 3, 224, 224])
Label batch shape: torch.Size([16])
Labels: tensor([0, 2, 4, 1, 1, 4, 3, 0, 2, 1, 0, 0, 0, 4, 1, 4])


In [48]:
# Fine-tuning learning rate
transfer_optimizer = torch.optim.Adam(
    transfer_model.parameters(),
    lr=1e-5
)

# Cross-entropy loss
transfer_criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

print("Transfer-learning optimizer created successfully!")
print("Learning rate: 1e-5")

Transfer-learning optimizer created successfully!
Learning rate: 1e-5


In [49]:
# Calculate IDRiD-specific class weights
idrid_counts = train_df["label"].value_counts().sort_index()

idrid_weights = 1.0 / torch.tensor(
    idrid_counts.values,
    dtype=torch.float32
)

# Normalize average weight to 1
idrid_weights = idrid_weights / idrid_weights.mean()

idrid_weights = idrid_weights.to(device)

# Correct IDRiD loss
transfer_criterion = nn.CrossEntropyLoss(
    weight=idrid_weights
)

# Fine-tuning optimizer
transfer_optimizer = torch.optim.Adam(
    transfer_model.parameters(),
    lr=1e-5
)

print("Correct IDRiD transfer-learning setup ready!")
print("IDRiD class weights:", idrid_weights)
print("Learning rate:", 1e-5)

Correct IDRiD transfer-learning setup ready!
IDRiD class weights: tensor([0.3780, 2.5281, 0.3711, 0.6856, 1.0372])
Learning rate: 1e-05


In [50]:
transfer_train_loss, transfer_train_acc = train_one_epoch(
    transfer_model,
    idrid_train_loader_balanced,
    transfer_criterion,
    transfer_optimizer,
    device
)

print("\n========== IDRiD TRANSFER EPOCH 1 COMPLETE ==========")
print(f"Train Loss: {transfer_train_loss:.4f}")
print(f"Train Accuracy: {transfer_train_acc:.2f}%")

Batch 20/21 | Loss: 1.7235 | Accuracy: 41.88%
Batch 21/21 | Loss: 0.7325 | Accuracy: 42.73%

========== IDRiD TRANSFER EPOCH 1 COMPLETE ==========
Train Loss: 1.3952
Train Accuracy: 42.73%


In [51]:
transfer_val_loss, transfer_val_acc = validate_one_epoch(
    transfer_model,
    idrid_val_loader,
    transfer_criterion,
    device
)

print("\n========== IDRiD TRANSFER EPOCH 1 VALIDATION ==========")
print(f"Validation Loss: {transfer_val_loss:.4f}")
print(f"Validation Accuracy: {transfer_val_acc:.2f}%")


========== IDRiD TRANSFER EPOCH 1 VALIDATION ==========
Validation Loss: 1.3434
Validation Accuracy: 38.55%


In [52]:
transfer_train_loss, transfer_train_acc = train_one_epoch(
    transfer_model,
    idrid_train_loader_balanced,
    transfer_criterion,
    transfer_optimizer,
    device
)

print("\n========== IDRiD TRANSFER EPOCH 2 COMPLETE ==========")
print(f"Train Loss: {transfer_train_loss:.4f}")
print(f"Train Accuracy: {transfer_train_acc:.2f}%")

Batch 20/21 | Loss: 1.3033 | Accuracy: 39.38%
Batch 21/21 | Loss: 1.1082 | Accuracy: 39.09%

========== IDRiD TRANSFER EPOCH 2 COMPLETE ==========
Train Loss: 1.3303
Train Accuracy: 39.09%


In [53]:
transfer_val_loss, transfer_val_acc = validate_one_epoch(
    transfer_model,
    idrid_val_loader,
    transfer_criterion,
    device
)

print("\n========== IDRiD TRANSFER EPOCH 2 VALIDATION ==========")
print(f"Validation Loss: {transfer_val_loss:.4f}")
print(f"Validation Accuracy: {transfer_val_acc:.2f}%")


========== IDRiD TRANSFER EPOCH 2 VALIDATION ==========
Validation Loss: 1.3503
Validation Accuracy: 38.55%


In [54]:
import torch
import torch.nn as nn
from torchvision.models import efficientnet_b0

# Create fresh EfficientNet-B0
transfer_partial_model = efficientnet_b0(weights=None)

num_features = transfer_partial_model.classifier[1].in_features
transfer_partial_model.classifier[1] = nn.Linear(num_features, 5)

# Load APTOS-trained weights
checkpoint = torch.load(
    "models/retina_xai_aptos_efficientnet_b0.pth",
    map_location=device
)

transfer_partial_model.load_state_dict(checkpoint["model_state_dict"])

# Freeze everything first
for param in transfer_partial_model.parameters():
    param.requires_grad = False

# Unfreeze the last feature block
for param in transfer_partial_model.features[-1].parameters():
    param.requires_grad = True

# Unfreeze classifier
for param in transfer_partial_model.classifier.parameters():
    param.requires_grad = True

transfer_partial_model = transfer_partial_model.to(device)

print("Partial transfer model ready!")

Partial transfer model ready!


In [55]:
total_params = sum(
    p.numel() for p in transfer_partial_model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in transfer_partial_model.parameters()
    if p.requires_grad
)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Total parameters: 4,013,953
Trainable parameters: 418,565


In [56]:
transfer_partial_criterion = nn.CrossEntropyLoss(
    weight=idrid_weights
)

transfer_partial_optimizer = torch.optim.Adam(
    filter(
        lambda p: p.requires_grad,
        transfer_partial_model.parameters()
    ),
    lr=1e-4
)

print("Partial transfer-learning setup ready!")
print(f"Learning rate: {1e-4}")

Partial transfer-learning setup ready!
Learning rate: 0.0001


In [57]:
partial_train_loss, partial_train_acc = train_one_epoch(
    transfer_partial_model,
    idrid_train_loader_balanced,
    transfer_partial_criterion,
    transfer_partial_optimizer,
    device
)

print("\n========== PARTIAL TRANSFER EPOCH 1 COMPLETE ==========")
print(f"Train Loss: {partial_train_loss:.4f}")
print(f"Train Accuracy: {partial_train_acc:.2f}%")

Batch 20/21 | Loss: 0.6727 | Accuracy: 44.38%
Batch 21/21 | Loss: 0.8188 | Accuracy: 44.55%

========== PARTIAL TRANSFER EPOCH 1 COMPLETE ==========
Train Loss: 1.2467
Train Accuracy: 44.55%


In [58]:
partial_val_loss, partial_val_acc = validate_one_epoch(
    transfer_partial_model,
    idrid_val_loader,
    transfer_partial_criterion,
    device
)

print("\n========== PARTIAL TRANSFER EPOCH 1 VALIDATION ==========")
print(f"Validation Loss: {partial_val_loss:.4f}")
print(f"Validation Accuracy: {partial_val_acc:.2f}%")


========== PARTIAL TRANSFER EPOCH 1 VALIDATION ==========
Validation Loss: 1.4625
Validation Accuracy: 33.73%


In [59]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, reduction="mean"):
        super().__init__()
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = nn.functional.cross_entropy(
            inputs,
            targets,
            reduction="none"
        )

        pt = torch.exp(-ce_loss)

        focal_loss = (1 - pt) ** self.gamma * ce_loss

        if self.reduction == "mean":
            return focal_loss.mean()
        elif self.reduction == "sum":
            return focal_loss.sum()
        else:
            return focal_loss


print("Focal Loss defined successfully!")

Focal Loss defined successfully!


In [60]:
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

focal_model = efficientnet_b0(
    weights=EfficientNet_B0_Weights.DEFAULT
)

num_features = focal_model.classifier[1].in_features

focal_model.classifier[1] = nn.Linear(
    num_features,
    5
)

focal_model = focal_model.to(device)

print("Fresh EfficientNet-B0 created!")
print("Number of classes:", 5)
print("Device:", device)

Fresh EfficientNet-B0 created!
Number of classes: 5
Device: cpu


In [61]:
focal_criterion = FocalLoss(gamma=2.0)

focal_optimizer = torch.optim.Adam(
    focal_model.parameters(),
    lr=1e-4
)

print("Focal Loss training setup ready!")
print("Gamma:", 2.0)
print("Learning rate:", 1e-4)

Focal Loss training setup ready!
Gamma: 2.0
Learning rate: 0.0001


In [62]:
focal_train_loss, focal_train_acc = train_one_epoch(
    focal_model,
    idrid_train_loader_balanced,
    focal_criterion,
    focal_optimizer,
    device
)

print("\n========== FOCAL LOSS EPOCH 1 COMPLETE ==========")
print(f"Train Loss: {focal_train_loss:.4f}")
print(f"Train Accuracy: {focal_train_acc:.2f}%")

Batch 20/21 | Loss: 0.7605 | Accuracy: 33.44%
Batch 21/21 | Loss: 1.0531 | Accuracy: 33.33%

========== FOCAL LOSS EPOCH 1 COMPLETE ==========
Train Loss: 0.9575
Train Accuracy: 33.33%


In [63]:
focal_val_loss, focal_val_acc = validate_one_epoch(
    focal_model,
    idrid_val_loader,
    focal_criterion,
    device
)

print("\n========== FOCAL LOSS EPOCH 1 VALIDATION ==========")
print(f"Validation Loss: {focal_val_loss:.4f}")
print(f"Validation Accuracy: {focal_val_acc:.2f}%")


========== FOCAL LOSS EPOCH 1 VALIDATION ==========
Validation Loss: 0.8979
Validation Accuracy: 33.73%


In [64]:
focal_train_loss, focal_train_acc = train_one_epoch(
    focal_model,
    idrid_train_loader_balanced,
    focal_criterion,
    focal_optimizer,
    device
)

print("\n========== FOCAL LOSS EPOCH 2 COMPLETE ==========")
print(f"Train Loss: {focal_train_loss:.4f}")
print(f"Train Accuracy: {focal_train_acc:.2f}%")

Batch 20/21 | Loss: 0.6236 | Accuracy: 55.00%
Batch 21/21 | Loss: 0.5358 | Accuracy: 55.45%

========== FOCAL LOSS EPOCH 2 COMPLETE ==========
Train Loss: 0.7392
Train Accuracy: 55.45%


In [65]:
focal_val_loss, focal_val_acc = validate_one_epoch(
    focal_model,
    idrid_val_loader,
    focal_criterion,
    device
)

print("\n========== FOCAL LOSS EPOCH 2 VALIDATION ==========")
print(f"Validation Loss: {focal_val_loss:.4f}")
print(f"Validation Accuracy: {focal_val_acc:.2f}%")


========== FOCAL LOSS EPOCH 2 VALIDATION ==========
Validation Loss: 0.7660
Validation Accuracy: 45.78%


In [66]:
focal_train_loss, focal_train_acc = train_one_epoch(
    focal_model,
    idrid_train_loader_balanced,
    focal_criterion,
    focal_optimizer,
    device
)

print("\n========== FOCAL LOSS EPOCH 3 COMPLETE ==========")
print(f"Train Loss: {focal_train_loss:.4f}")
print(f"Train Accuracy: {focal_train_acc:.2f}%")

Batch 20/21 | Loss: 0.4300 | Accuracy: 59.69%
Batch 21/21 | Loss: 0.5073 | Accuracy: 60.30%

========== FOCAL LOSS EPOCH 3 COMPLETE ==========
Train Loss: 0.6059
Train Accuracy: 60.30%


In [67]:
focal_val_loss, focal_val_acc = validate_one_epoch(
    focal_model,
    idrid_val_loader,
    focal_criterion,
    device
)

print("\n========== FOCAL LOSS EPOCH 3 VALIDATION ==========")
print(f"Validation Loss: {focal_val_loss:.4f}")
print(f"Validation Accuracy: {focal_val_acc:.2f}%")


========== FOCAL LOSS EPOCH 3 VALIDATION ==========
Validation Loss: 0.6638
Validation Accuracy: 50.60%


In [68]:
focal_train_loss, focal_train_acc = train_one_epoch(
    focal_model,
    idrid_train_loader_balanced,
    focal_criterion,
    focal_optimizer,
    device
)

print("\n========== FOCAL LOSS EPOCH 4 COMPLETE ==========")
print(f"Train Loss: {focal_train_loss:.4f}")
print(f"Train Accuracy: {focal_train_acc:.2f}%")

Batch 20/21 | Loss: 0.6823 | Accuracy: 65.31%
Batch 21/21 | Loss: 0.5555 | Accuracy: 65.15%

========== FOCAL LOSS EPOCH 4 COMPLETE ==========
Train Loss: 0.5197
Train Accuracy: 65.15%


In [69]:
focal_val_loss, focal_val_acc = validate_one_epoch(
    focal_model,
    idrid_val_loader,
    focal_criterion,
    device
)

print("\n========== FOCAL LOSS EPOCH 4 VALIDATION ==========")
print(f"Validation Loss: {focal_val_loss:.4f}")
print(f"Validation Accuracy: {focal_val_acc:.2f}%")


========== FOCAL LOSS EPOCH 4 VALIDATION ==========
Validation Loss: 0.5837
Validation Accuracy: 56.63%


In [70]:
focal_train_loss, focal_train_acc = train_one_epoch(
    focal_model,
    idrid_train_loader_balanced,
    focal_criterion,
    focal_optimizer,
    device
)

print("\n========== FOCAL LOSS EPOCH 5 COMPLETE ==========")
print(f"Train Loss: {focal_train_loss:.4f}")
print(f"Train Accuracy: {focal_train_acc:.2f}%")

Batch 20/21 | Loss: 0.4136 | Accuracy: 72.50%
Batch 21/21 | Loss: 0.3477 | Accuracy: 72.12%

========== FOCAL LOSS EPOCH 5 COMPLETE ==========
Train Loss: 0.4140
Train Accuracy: 72.12%


In [71]:
focal_val_loss, focal_val_acc = validate_one_epoch(
    focal_model,
    idrid_val_loader,
    focal_criterion,
    device
)

print("\n========== FOCAL LOSS EPOCH 5 VALIDATION ==========")
print(f"Validation Loss: {focal_val_loss:.4f}")
print(f"Validation Accuracy: {focal_val_acc:.2f}%")


========== FOCAL LOSS EPOCH 5 VALIDATION ==========
Validation Loss: 0.5006
Validation Accuracy: 55.42%


In [72]:
focal_train_loss, focal_train_acc = train_one_epoch(
    focal_model,
    idrid_train_loader_balanced,
    focal_criterion,
    focal_optimizer,
    device
)

print("\n========== FOCAL LOSS EPOCH 6 COMPLETE ==========")
print(f"Train Loss: {focal_train_loss:.4f}")
print(f"Train Accuracy: {focal_train_acc:.2f}%")

Batch 20/21 | Loss: 0.3968 | Accuracy: 70.31%
Batch 21/21 | Loss: 0.3648 | Accuracy: 70.30%

========== FOCAL LOSS EPOCH 6 COMPLETE ==========
Train Loss: 0.3948
Train Accuracy: 70.30%


In [73]:
focal_val_loss, focal_val_acc = validate_one_epoch(
    focal_model,
    idrid_val_loader,
    focal_criterion,
    device
)

print("\n========== FOCAL LOSS EPOCH 6 VALIDATION ==========")
print(f"Validation Loss: {focal_val_loss:.4f}")
print(f"Validation Accuracy: {focal_val_acc:.2f}%")


========== FOCAL LOSS EPOCH 6 VALIDATION ==========
Validation Loss: 0.4782
Validation Accuracy: 60.24%


In [74]:
FOCAL_MODEL_PATH = "models/retina_xai_focal_efficientnet_b0_best.pth"

torch.save({
    "model_state_dict": focal_model.state_dict(),
    "num_classes": 5,
    "class_names": {
        0: "No DR",
        1: "Mild",
        2: "Moderate",
        3: "Severe",
        4: "Proliferative"
    },
    "epoch": 6,
    "val_accuracy": focal_val_acc,
    "val_loss": focal_val_loss,
    "gamma": 2.0
}, FOCAL_MODEL_PATH)

print("Best Focal Loss model saved successfully!")
print(f"Epoch: 6")
print(f"Validation Accuracy: {focal_val_acc:.2f}%")
print(f"Validation Loss: {focal_val_loss:.4f}")

Best Focal Loss model saved successfully!
Epoch: 6
Validation Accuracy: 60.24%
Validation Loss: 0.4782


In [75]:
focal_train_loss, focal_train_acc = train_one_epoch(
    focal_model,
    idrid_train_loader_balanced,
    focal_criterion,
    focal_optimizer,
    device
)

print("\n========== FOCAL LOSS EPOCH 7 COMPLETE ==========")
print(f"Train Loss: {focal_train_loss:.4f}")
print(f"Train Accuracy: {focal_train_acc:.2f}%")

Batch 20/21 | Loss: 0.2435 | Accuracy: 76.56%
Batch 21/21 | Loss: 0.5220 | Accuracy: 76.36%

========== FOCAL LOSS EPOCH 7 COMPLETE ==========
Train Loss: 0.3117
Train Accuracy: 76.36%


In [76]:
focal_val_loss, focal_val_acc = validate_one_epoch(
    focal_model,
    idrid_val_loader,
    focal_criterion,
    device
)

print("\n========== FOCAL LOSS EPOCH 7 VALIDATION ==========")
print(f"Validation Loss: {focal_val_loss:.4f}")
print(f"Validation Accuracy: {focal_val_acc:.2f}%")


========== FOCAL LOSS EPOCH 7 VALIDATION ==========
Validation Loss: 0.4517
Validation Accuracy: 62.65%


In [77]:
FOCAL_MODEL_PATH = "models/retina_xai_focal_efficientnet_b0_best.pth"

torch.save({
    "model_state_dict": focal_model.state_dict(),
    "num_classes": 5,
    "class_names": {
        0: "No DR",
        1: "Mild",
        2: "Moderate",
        3: "Severe",
        4: "Proliferative"
    },
    "epoch": 7,
    "val_accuracy": focal_val_acc,
    "val_loss": focal_val_loss,
    "gamma": 2.0
}, FOCAL_MODEL_PATH)

print("Best Focal Loss model updated!")
print(f"Epoch: 7")
print(f"Validation Accuracy: {focal_val_acc:.2f}%")
print(f"Validation Loss: {focal_val_loss:.4f}")

Best Focal Loss model updated!
Epoch: 7
Validation Accuracy: 62.65%
Validation Loss: 0.4517


In [79]:
focal_train_loss, focal_train_acc = train_one_epoch(
    focal_model,
    idrid_train_loader_balanced,
    focal_criterion,
    focal_optimizer,
    device
)

print("\n========== FOCAL LOSS EPOCH 8 COMPLETE ==========")
print(f"Train Loss: {focal_train_loss:.4f}")
print(f"Train Accuracy: {focal_train_acc:.2f}%")

Batch 20/21 | Loss: 0.2972 | Accuracy: 79.69%
Batch 21/21 | Loss: 0.3320 | Accuracy: 79.39%

========== FOCAL LOSS EPOCH 8 COMPLETE ==========
Train Loss: 0.2656
Train Accuracy: 79.39%


In [80]:
focal_val_loss, focal_val_acc = validate_one_epoch(
    focal_model,
    idrid_val_loader,
    focal_criterion,
    device
)

print("\n========== FOCAL LOSS EPOCH 8 VALIDATION ==========")
print(f"Validation Loss: {focal_val_loss:.4f}")
print(f"Validation Accuracy: {focal_val_acc:.2f}%")


========== FOCAL LOSS EPOCH 8 VALIDATION ==========
Validation Loss: 0.4679
Validation Accuracy: 62.65%


In [81]:
focal_train_loss, focal_train_acc = train_one_epoch(
    focal_model,
    idrid_train_loader_balanced,
    focal_criterion,
    focal_optimizer,
    device
)

print("\n========== FOCAL LOSS EPOCH 9 COMPLETE ==========")
print(f"Train Loss: {focal_train_loss:.4f}")
print(f"Train Accuracy: {focal_train_acc:.2f}%")

Batch 20/21 | Loss: 0.3079 | Accuracy: 83.44%
Batch 21/21 | Loss: 0.1156 | Accuracy: 83.64%

========== FOCAL LOSS EPOCH 9 COMPLETE ==========
Train Loss: 0.2378
Train Accuracy: 83.64%


In [82]:
focal_val_loss, focal_val_acc = validate_one_epoch(
    focal_model,
    idrid_val_loader,
    focal_criterion,
    device
)

print("\n========== FOCAL LOSS EPOCH 9 VALIDATION ==========")
print(f"Validation Loss: {focal_val_loss:.4f}")
print(f"Validation Accuracy: {focal_val_acc:.2f}%")


========== FOCAL LOSS EPOCH 9 VALIDATION ==========
Validation Loss: 0.4411
Validation Accuracy: 61.45%


In [83]:
# Load the best Focal Loss checkpoint
best_checkpoint = torch.load(
    "models/retina_xai_focal_efficientnet_b0_best.pth",
    map_location=device
)

focal_model.load_state_dict(best_checkpoint["model_state_dict"])
focal_model = focal_model.to(device)
focal_model.eval()

print("Best Focal Loss model loaded!")
print(f"Epoch: {best_checkpoint['epoch']}")
print(f"Validation Accuracy: {best_checkpoint['val_accuracy']:.2f}%")
print(f"Validation Loss: {best_checkpoint['val_loss']:.4f}")

Best Focal Loss model loaded!
Epoch: 7
Validation Accuracy: 62.65%
Validation Loss: 0.4517


In [86]:
import os
import pandas as pd

BASE_DIR = r"C:\Users\vu241\OneDrive\Desktop\Diabetic-Retinopathy-Screening\Disease Dataset\B. Disease Grading"

TEST_DIR = os.path.join(
    BASE_DIR,
    "1. Original Images",
    "b. Testing Set"
)

GROUNDTRUTH_DIR = os.path.join(
    BASE_DIR,
    "2. Groundtruths"
)

TEST_LABEL_FILE = os.path.join(
    GROUNDTRUTH_DIR,
    "b. IDRiD_Disease Grading_Testing Labels.csv"
)

print("TEST_DIR:", TEST_DIR)
print("Test folder exists:", os.path.exists(TEST_DIR))
print("Label file exists:", os.path.exists(TEST_LABEL_FILE))

TEST_DIR: C:\Users\vu241\OneDrive\Desktop\Diabetic-Retinopathy-Screening\Disease Dataset\B. Disease Grading\1. Original Images\b. Testing Set
Test folder exists: True
Label file exists: True


In [87]:
test_df = pd.read_csv(TEST_LABEL_FILE)

test_df.columns = test_df.columns.str.strip()

test_df["image_path"] = test_df["Image name"].apply(
    lambda x: os.path.join(TEST_DIR, x + ".jpg")
)

test_df["label"] = test_df["Retinopathy grade"].astype(int)

test_df = test_df[
    ["Image name", "image_path", "label"]
].copy()

print("Test images:", len(test_df))

print(
    "Missing images:",
    test_df["image_path"].apply(
        lambda x: not os.path.exists(x)
    ).sum()
)

print("\nTest class distribution:")
print(test_df["label"].value_counts().sort_index())

Test images: 103
Missing images: 0

Test class distribution:
label
0    34
1     5
2    32
3    19
4    13
Name: count, dtype: int64


In [88]:
test_dataset = PreprocessedRetinaDataset(
    test_df,
    transform=val_transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0
)

print("Test loader created successfully!")
print("Number of test images:", len(test_dataset))
print("Number of batches:", len(test_loader))

Test loader created successfully!
Number of test images: 103
Number of batches: 7


In [89]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

all_preds = []
all_labels = []

focal_model.eval()

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)

        outputs = focal_model(images)
        predictions = torch.argmax(outputs, dim=1)

        all_preds.extend(predictions.cpu().numpy())
        all_labels.extend(labels.numpy())

test_accuracy = accuracy_score(all_labels, all_preds)

print("\n========== FOCAL LOSS TEST RESULTS ==========")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

print("\nClassification Report:")
print(
    classification_report(
        all_labels,
        all_preds,
        target_names=[
            "No DR",
            "Mild",
            "Moderate",
            "Severe",
            "Proliferative"
        ],
        zero_division=0
    )
)

print("\nConfusion Matrix:")
print(confusion_matrix(all_labels, all_preds))


========== FOCAL LOSS TEST RESULTS ==========
Test Accuracy: 47.57%

Classification Report:
               precision    recall  f1-score   support

        No DR       0.63      0.71      0.67        34
         Mild       0.00      0.00      0.00         5
     Moderate       0.44      0.44      0.44        32
       Severe       0.38      0.32      0.34        19
Proliferative       0.36      0.38      0.37        13

     accuracy                           0.48       103
    macro avg       0.36      0.37      0.36       103
 weighted avg       0.46      0.48      0.47       103


Confusion Matrix:
[[24  3  6  1  0]
 [ 3  0  2  0  0]
 [ 8  0 14  7  3]
 [ 1  0  6  6  6]
 [ 2  0  4  2  5]]


In [90]:
# Combined 

In [91]:
# ============================================
# COMBINED DATASET - STEP 1
# Create APTOS + IDRiD training dataframe
# ============================================

# IDRiD training data
idrid_combined_df = df.copy()

idrid_combined_df = idrid_combined_df[
    ["image_path", "label"]
].copy()

# APTOS training data
aptos_combined_df = aptos_train_df[
    ["image_path", "label"]
].copy()

# Add dataset identifier
aptos_combined_df["dataset"] = "APTOS"
idrid_combined_df["dataset"] = "IDRiD"

# Combine
combined_train_df = pd.concat(
    [
        aptos_combined_df,
        idrid_combined_df
    ],
    ignore_index=True
)

print("========== COMBINED DATASET ==========")
print("APTOS training images:", len(aptos_combined_df))
print("IDRiD training images:", len(idrid_combined_df))
print("Total training images:", len(combined_train_df))

print("\nClass distribution:")
print(
    combined_train_df["label"]
    .value_counts()
    .sort_index()
)

========== COMBINED DATASET ==========
APTOS training images: 2929
IDRiD training images: 413
Total training images: 3342

Class distribution:
label
0    1578
1     316
2     935
3     228
4     285
Name: count, dtype: int64


In [92]:
# ============================================
# COMBINED DATASET - STEP 2
# Dataset + Balanced Sampler
# ============================================

combined_train_dataset = PreprocessedRetinaDataset(
    combined_train_df,
    transform=train_transform
)

# Calculate inverse-frequency sampling weights
combined_class_counts = (
    combined_train_df["label"]
    .value_counts()
    .sort_index()
)

combined_class_sample_weights = (
    1.0 / combined_class_counts.values
)

sample_weights = combined_train_df["label"].map(
    lambda label: combined_class_sample_weights[label]
).values

sample_weights = torch.tensor(
    sample_weights,
    dtype=torch.double
)

combined_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

combined_train_loader = DataLoader(
    combined_train_dataset,
    batch_size=16,
    sampler=combined_sampler,
    num_workers=0
)

print("========== COMBINED DATALOADER ==========")
print("Dataset size:", len(combined_train_dataset))
print("Batches per epoch:", len(combined_train_loader))
print("Batch size:", 16)
print("Balanced sampler: Enabled")

========== COMBINED DATALOADER ==========
Dataset size: 3342
Batches per epoch: 209
Batch size: 16
Balanced sampler: Enabled


In [93]:
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
import torch.nn as nn

combined_model = efficientnet_b0(
    weights=EfficientNet_B0_Weights.DEFAULT
)

num_features = combined_model.classifier[1].in_features

combined_model.classifier[1] = nn.Linear(
    num_features,
    5
)

combined_model = combined_model.to(device)

print("========== COMBINED MODEL ==========")
print("Model: EfficientNet-B0")
print("Classes: 5")
print("Device:", device)

========== COMBINED MODEL ==========
Model: EfficientNet-B0
Classes: 5
Device: cpu


In [94]:
combined_criterion = nn.CrossEntropyLoss()

combined_optimizer = torch.optim.Adam(
    combined_model.parameters(),
    lr=1e-4
)

print("========== TRAINING SETUP ==========")
print("Loss: CrossEntropyLoss")
print("Learning rate:", 1e-4)

========== TRAINING SETUP ==========
Loss: CrossEntropyLoss
Learning rate: 0.0001


In [95]:
combined_images, combined_labels = next(iter(combined_train_loader))

print("Image batch shape:", combined_images.shape)
print("Label batch shape:", combined_labels.shape)
print("Labels:", combined_labels.tolist())
print("Unique classes in batch:", torch.unique(combined_labels).tolist())

Image batch shape: torch.Size([16, 3, 224, 224])
Label batch shape: torch.Size([16])
Labels: [1, 1, 2, 4, 2, 3, 2, 2, 3, 2, 4, 1, 3, 0, 0, 3]
Unique classes in batch: [0, 1, 2, 3, 4]


In [96]:
combined_train_loss, combined_train_acc = train_one_epoch(
    combined_model,
    combined_train_loader,
    combined_criterion,
    combined_optimizer,
    device
)

print("\n========== COMBINED TRAINING EPOCH 1 COMPLETE ==========")
print(f"Train Loss: {combined_train_loss:.4f}")
print(f"Train Accuracy: {combined_train_acc:.2f}%")

Batch 20/209 | Loss: 1.3632 | Accuracy: 31.25%
Batch 40/209 | Loss: 1.4629 | Accuracy: 38.28%
Batch 60/209 | Loss: 1.0964 | Accuracy: 42.29%
Batch 80/209 | Loss: 0.9768 | Accuracy: 44.30%
Batch 100/209 | Loss: 0.9482 | Accuracy: 47.00%
Batch 120/209 | Loss: 0.9683 | Accuracy: 48.70%
Batch 140/209 | Loss: 1.0008 | Accuracy: 49.87%
Batch 160/209 | Loss: 0.7792 | Accuracy: 51.13%
Batch 180/209 | Loss: 1.2261 | Accuracy: 52.33%
Batch 200/209 | Loss: 0.8468 | Accuracy: 53.12%
Batch 209/209 | Loss: 0.9224 | Accuracy: 53.95%

========== COMBINED TRAINING EPOCH 1 COMPLETE ==========
Train Loss: 1.1320
Train Accuracy: 53.95%


In [97]:
combined_val_loss, combined_val_acc = validate_one_epoch(
    combined_model,
    idrid_val_loader,
    combined_criterion,
    device
)

print("\n========== COMBINED EPOCH 1 VALIDATION ==========")
print(f"Validation Loss: {combined_val_loss:.4f}")
print(f"Validation Accuracy: {combined_val_acc:.2f}%")


========== COMBINED EPOCH 1 VALIDATION ==========
Validation Loss: 1.0617
Validation Accuracy: 54.22%


In [98]:
combined_train_loss, combined_train_acc = train_one_epoch(
    combined_model,
    combined_train_loader,
    combined_criterion,
    combined_optimizer,
    device
)

print("\n========== COMBINED TRAINING EPOCH 2 COMPLETE ==========")
print(f"Train Loss: {combined_train_loss:.4f}")
print(f"Train Accuracy: {combined_train_acc:.2f}%")

Batch 20/209 | Loss: 0.9626 | Accuracy: 67.50%
Batch 40/209 | Loss: 0.6794 | Accuracy: 68.44%
Batch 60/209 | Loss: 0.9596 | Accuracy: 68.12%
Batch 80/209 | Loss: 0.7659 | Accuracy: 67.58%
Batch 100/209 | Loss: 1.2138 | Accuracy: 67.88%
Batch 120/209 | Loss: 0.6038 | Accuracy: 68.80%
Batch 140/209 | Loss: 0.9895 | Accuracy: 69.42%
Batch 160/209 | Loss: 0.7375 | Accuracy: 69.30%
Batch 180/209 | Loss: 0.6071 | Accuracy: 69.06%
Batch 200/209 | Loss: 0.6913 | Accuracy: 69.72%
Batch 209/209 | Loss: 0.5038 | Accuracy: 69.78%

========== COMBINED TRAINING EPOCH 2 COMPLETE ==========
Train Loss: 0.7797
Train Accuracy: 69.78%


In [99]:
combined_val_loss, combined_val_acc = validate_one_epoch(
    combined_model,
    idrid_val_loader,
    combined_criterion,
    device
)

print("\n========== COMBINED EPOCH 2 VALIDATION ==========")
print(f"Validation Loss: {combined_val_loss:.4f}")
print(f"Validation Accuracy: {combined_val_acc:.2f}%")


========== COMBINED EPOCH 2 VALIDATION ==========
Validation Loss: 0.9245
Validation Accuracy: 61.45%


In [100]:
combined_train_loss, combined_train_acc = train_one_epoch(
    combined_model,
    combined_train_loader,
    combined_criterion,
    combined_optimizer,
    device
)

print("\n========== COMBINED TRAINING EPOCH 3 COMPLETE ==========")
print(f"Train Loss: {combined_train_loss:.4f}")
print(f"Train Accuracy: {combined_train_acc:.2f}%")

Batch 20/209 | Loss: 0.6434 | Accuracy: 67.81%
Batch 40/209 | Loss: 0.9480 | Accuracy: 68.75%
Batch 60/209 | Loss: 0.6979 | Accuracy: 69.79%
Batch 80/209 | Loss: 0.4479 | Accuracy: 70.62%
Batch 100/209 | Loss: 0.5098 | Accuracy: 71.44%
Batch 120/209 | Loss: 0.6396 | Accuracy: 72.08%
Batch 140/209 | Loss: 0.8970 | Accuracy: 72.77%
Batch 160/209 | Loss: 0.4132 | Accuracy: 73.55%
Batch 180/209 | Loss: 0.5698 | Accuracy: 73.75%
Batch 200/209 | Loss: 0.8059 | Accuracy: 74.25%
Batch 209/209 | Loss: 0.4748 | Accuracy: 74.21%

========== COMBINED TRAINING EPOCH 3 COMPLETE ==========
Train Loss: 0.6586
Train Accuracy: 74.21%


In [101]:
combined_val_loss, combined_val_acc = validate_one_epoch(
    combined_model,
    idrid_val_loader,
    combined_criterion,
    device
)

print("\n========== COMBINED EPOCH 3 VALIDATION ==========")
print(f"Validation Loss: {combined_val_loss:.4f}")
print(f"Validation Accuracy: {combined_val_acc:.2f}%")


========== COMBINED EPOCH 3 VALIDATION ==========
Validation Loss: 0.8949
Validation Accuracy: 61.45%


In [102]:
combined_train_loss, combined_train_acc = train_one_epoch(
    combined_model,
    combined_train_loader,
    combined_criterion,
    combined_optimizer,
    device
)

print("\n========== COMBINED TRAINING EPOCH 4 COMPLETE ==========")
print(f"Train Loss: {combined_train_loss:.4f}")
print(f"Train Accuracy: {combined_train_acc:.2f}%")

Batch 20/209 | Loss: 0.5066 | Accuracy: 75.00%
Batch 40/209 | Loss: 0.4795 | Accuracy: 76.72%
Batch 60/209 | Loss: 0.5021 | Accuracy: 77.81%
Batch 80/209 | Loss: 0.9431 | Accuracy: 78.20%
Batch 100/209 | Loss: 1.0373 | Accuracy: 78.38%
Batch 120/209 | Loss: 0.4071 | Accuracy: 78.49%
Batch 140/209 | Loss: 0.7359 | Accuracy: 78.84%
Batch 160/209 | Loss: 0.7395 | Accuracy: 78.79%
Batch 180/209 | Loss: 0.6118 | Accuracy: 79.24%
Batch 200/209 | Loss: 0.6020 | Accuracy: 79.41%
Batch 209/209 | Loss: 0.2057 | Accuracy: 79.71%

========== COMBINED TRAINING EPOCH 4 COMPLETE ==========
Train Loss: 0.5289
Train Accuracy: 79.71%


In [103]:
combined_val_loss, combined_val_acc = validate_one_epoch(
    combined_model,
    idrid_val_loader,
    combined_criterion,
    device
)

print("\n========== COMBINED EPOCH 4 VALIDATION ==========")
print(f"Validation Loss: {combined_val_loss:.4f}")
print(f"Validation Accuracy: {combined_val_acc:.2f}%")


========== COMBINED EPOCH 4 VALIDATION ==========
Validation Loss: 0.5268
Validation Accuracy: 83.13%


In [104]:
COMBINED_MODEL_PATH = "models/retina_xai_combined_efficientnet_b0_best.pth"

torch.save({
    "model_state_dict": combined_model.state_dict(),
    "num_classes": 5,
    "class_names": {
        0: "No DR",
        1: "Mild",
        2: "Moderate",
        3: "Severe",
        4: "Proliferative"
    },
    "epoch": 4,
    "val_accuracy": combined_val_acc,
    "val_loss": combined_val_loss,
    "training_datasets": ["APTOS 2019", "IDRiD"],
    "preprocessing": "fundus_crop_resize"
}, COMBINED_MODEL_PATH)

print("Best combined model saved!")
print(f"Epoch: 4")
print(f"Validation Accuracy: {combined_val_acc:.2f}%")
print(f"Validation Loss: {combined_val_loss:.4f}")

Best combined model saved!
Epoch: 4
Validation Accuracy: 83.13%
Validation Loss: 0.5268


In [105]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

combined_model.eval()

val_preds = []
val_labels = []

with torch.no_grad():
    for images, labels in idrid_val_loader:
        images = images.to(device)

        outputs = combined_model(images)
        predictions = torch.argmax(outputs, dim=1)

        val_preds.extend(predictions.cpu().numpy())
        val_labels.extend(labels.numpy())

# Accuracy
val_accuracy = accuracy_score(val_labels, val_preds)

# Macro F1
macro_f1 = f1_score(
    val_labels,
    val_preds,
    average="macro",
    zero_division=0
)

# Weighted F1
weighted_f1 = f1_score(
    val_labels,
    val_preds,
    average="weighted",
    zero_division=0
)

print("\n========== COMBINED MODEL VALIDATION ==========")
print(f"Validation Accuracy: {val_accuracy * 100:.2f}%")
print(f"Macro F1: {macro_f1:.4f}")
print(f"Weighted F1: {weighted_f1:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        val_labels,
        val_preds,
        target_names=[
            "No DR",
            "Mild",
            "Moderate",
            "Severe",
            "Proliferative"
        ],
        zero_division=0
    )
)

print("\nConfusion Matrix:")
print(confusion_matrix(val_labels, val_preds))


========== COMBINED MODEL VALIDATION ==========
Validation Accuracy: 83.13%
Macro F1: 0.8114
Weighted F1: 0.8208

Classification Report:
               precision    recall  f1-score   support

        No DR       0.82      1.00      0.90        27
         Mild       0.60      0.75      0.67         4
     Moderate       1.00      0.56      0.71        27
       Severe       0.74      0.93      0.82        15
Proliferative       0.91      1.00      0.95        10

     accuracy                           0.83        83
    macro avg       0.81      0.85      0.81        83
 weighted avg       0.86      0.83      0.82        83


Confusion Matrix:
[[27  0  0  0  0]
 [ 1  3  0  0  0]
 [ 5  2 15  5  0]
 [ 0  0  0 14  1]
 [ 0  0  0  0 10]]


In [106]:
best_combined_checkpoint = torch.load(
    "models/retina_xai_combined_efficientnet_b0_best.pth",
    map_location=device
)

combined_model.load_state_dict(
    best_combined_checkpoint["model_state_dict"]
)

combined_model = combined_model.to(device)
combined_model.eval()

print("Best combined model loaded!")
print(f"Epoch: {best_combined_checkpoint['epoch']}")
print(
    f"Validation Accuracy: "
    f"{best_combined_checkpoint['val_accuracy']:.2f}%"
)
print(
    f"Validation Loss: "
    f"{best_combined_checkpoint['val_loss']:.4f}"
)

Best combined model loaded!
Epoch: 4
Validation Accuracy: 83.13%
Validation Loss: 0.5268


In [107]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

all_preds = []
all_labels = []

combined_model.eval()

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)

        outputs = combined_model(images)
        predictions = torch.argmax(outputs, dim=1)

        all_preds.extend(predictions.cpu().numpy())
        all_labels.extend(labels.numpy())

# Accuracy
test_accuracy = accuracy_score(
    all_labels,
    all_preds
)

# F1 scores
macro_f1 = f1_score(
    all_labels,
    all_preds,
    average="macro",
    zero_division=0
)

weighted_f1 = f1_score(
    all_labels,
    all_preds,
    average="weighted",
    zero_division=0
)

print("\n========== COMBINED MODEL — FINAL IDRiD TEST ==========")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")
print(f"Macro F1: {macro_f1:.4f}")
print(f"Weighted F1: {weighted_f1:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        all_labels,
        all_preds,
        target_names=[
            "No DR",
            "Mild",
            "Moderate",
            "Severe",
            "Proliferative"
        ],
        zero_division=0
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        all_labels,
        all_preds
    )
)


========== COMBINED MODEL — FINAL IDRiD TEST ==========
Test Accuracy: 57.28%
Macro F1: 0.4686
Weighted F1: 0.5720

Classification Report:
               precision    recall  f1-score   support

        No DR       0.61      0.74      0.67        34
         Mild       0.00      0.00      0.00         5
     Moderate       0.78      0.44      0.56        32
       Severe       0.56      0.74      0.64        19
Proliferative       0.50      0.46      0.48        13

     accuracy                           0.57       103
    macro avg       0.49      0.47      0.47       103
 weighted avg       0.61      0.57      0.57       103


Confusion Matrix:
[[25  6  0  3  0]
 [ 5  0  0  0  0]
 [ 8  1 14  5  4]
 [ 2  0  1 14  2]
 [ 1  0  3  3  6]]


In [1]:
# Lesion evidance

In [2]:
import os
from pathlib import Path

PROJECT_ROOT = Path(
    r"C:\Users\vu241\OneDrive\Desktop\Diabetic-Retinopathy-Screening"
)

SEG_ROOT = PROJECT_ROOT / "Disease Dataset" / "A. Segmentation"

TRAIN_IMAGES = SEG_ROOT / "1. Original Images" / "a. Training Set"
TEST_IMAGES = SEG_ROOT / "1. Original Images" / "b. Testing Set"

TRAIN_MASKS = SEG_ROOT / "2. All Segmentation Groundtruths" / "a. Training Set"
TEST_MASKS = SEG_ROOT / "2. All Segmentation Groundtruths" / "b. Testing Set"

print("Segmentation root:", SEG_ROOT)
print("Training images:", TRAIN_IMAGES)
print("Training masks:", TRAIN_MASKS)

print("\nPaths exist?")
print("TRAIN_IMAGES:", TRAIN_IMAGES.exists())
print("TEST_IMAGES :", TEST_IMAGES.exists())
print("TRAIN_MASKS :", TRAIN_MASKS.exists())
print("TEST_MASKS  :", TEST_MASKS.exists())

Segmentation root: C:\Users\vu241\OneDrive\Desktop\Diabetic-Retinopathy-Screening\Disease Dataset\A. Segmentation
Training images: C:\Users\vu241\OneDrive\Desktop\Diabetic-Retinopathy-Screening\Disease Dataset\A. Segmentation\1. Original Images\a. Training Set
Training masks: C:\Users\vu241\OneDrive\Desktop\Diabetic-Retinopathy-Screening\Disease Dataset\A. Segmentation\2. All Segmentation Groundtruths\a. Training Set

Paths exist?
TRAIN_IMAGES: True
TEST_IMAGES : True
TRAIN_MASKS : True
TEST_MASKS  : True
